# Trực quan hóa dữ liệu KLTN — hỗ trợ chọn ngân hàng (gửi tiết kiệm / vay)

Notebook này vẽ lại toàn bộ bộ biểu đồ mô tả trong `README.md` (mục *Trực quan hóa*) bằng
**matplotlib**, đọc thẳng từ mô hình sao `data/powerbi/` do `export_powerbi.py` sinh ra.

**Bốn cái bẫy trong README đã được cài sẵn vào code, không phải nhớ thủ công:**

1. `fact_deposit_rate` chồng 3 nguồn → mọi biểu đồ lãi suất tiền gửi đều **lọc `source` + `term_months` + `channel`**.
2. `fact_financial` có cả `freq=quarter` lẫn `freq=year` → notebook này dùng `fact_bank_quarterly`, không cộng chéo.
3. Không trộn chỉ tiêu của `fact_bank_quarterly` với `fact_financial` trong cùng một hình.
4. `dim_bank` có 53 dòng → mọi thứ đều lọc **`is_listed = True`** (22 mã).

Ba quy tắc vẽ được giữ xuyên suốt: **không dùng hai trục y** (hai đơn vị → hai panel),
**màu gắn với thực thể chứ không gắn với thứ hạng**, và **tối đa 3 chuỗi được tô màu** —
phần còn lại xám mờ làm nền, vì 22 màu thì không ai phân biệt nổi.

In [ ]:
from __future__ import annotations

import os
import textwrap
from pathlib import Path

import matplotlib as mpl
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D


def tim_goc(bat_dau: Path | None = None) -> Path:
    """Tìm gốc dự án — thư mục chứa data/powerbi — bằng cách đi ngược lên từ thư mục hiện tại.

    Nhờ vậy notebook chạy được dù đặt ở gốc dự án hay trong thư mục con (notebooks/, phan_tich/…),
    và không phụ thuộc vào chỗ Jupyter được khởi động.
    """
    bat_dau = (bat_dau or Path.cwd()).resolve()
    for d in [bat_dau, *bat_dau.parents]:
        if (d / "data" / "powerbi").is_dir():
            return d
    raise FileNotFoundError(textwrap.dedent(f"""
        Không thấy data/powerbi khi đi ngược lên từ {bat_dau}.
          - Chạy Jupyter trong trình duyệt (JupyterLite / jupyter.org/try-jupyter)? Môi trường đó
            KHÔNG đọc được ổ đĩa máy bạn. Phải chạy Jupyter cục bộ: mở terminal tại D:/KLTN rồi
            .venv/Scripts/jupyter lab
            (cài trước: .venv/Scripts/python -m pip install jupyterlab)
          - Chưa sinh mô hình sao? Chạy: python export_powerbi.py
          - Hoặc gán tay ngay dưới đây: ROOT = Path(r"D:/KLTN")
        """))


# Đặt biến môi trường KLTN_ROOT để ép đường dẫn, không thì tự dò
ROOT = Path(os.environ["KLTN_ROOT"]) if os.environ.get("KLTN_ROOT") else tim_goc()
PBI = ROOT / "data" / "powerbi"
FIGDIR = ROOT / "figures"
SAVE_FIG = True          # đặt False nếu không muốn ghi PNG ra figures/
if SAVE_FIG:
    FIGDIR.mkdir(exist_ok=True)

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)

print("Gốc dự án:", ROOT, "  (thư mục hiện tại:", Path.cwd(), ")")
print("Dữ liệu:", PBI)
print("Có mặt :", len(sorted(PBI.glob("*.csv"))), "file CSV")

## 0. Bảng màu và giao diện chung

Bảng màu phân loại dùng theo **thứ tự cố định** (slot 1 → 8), không bao giờ xoay vòng: người đọc
học "ngân hàng này màu xanh" thì lọc bớt ngân hàng khác cũng không được đổi màu của nó.
Thang **tuần tự** (một sắc, nhạt → đậm) dành cho độ lớn liên tục — heatmap. Thang **phân kỳ**
(đỏ ↔ xanh, giữa là xám) chỉ dùng khi số liệu có hai cực đối nhau — âm đỏ, dương xanh.

In [ ]:
SURFACE = "#fcfcfb"
INK = "#0b0b0b"
INK_2 = "#52514e"
INK_3 = "#7b7a75"
GRID = "#e8e7e3"
AXIS = "#d8d7d2"
MUTED = "#c9c8c3"          # đường nền cho các chuỗi không nhấn mạnh

SERIES = [
    "#2a78d6",  # 1 xanh dương
    "#eb6834",  # 2 cam
    "#1baf7a",  # 3 ngọc
    "#eda100",  # 4 vàng
    "#e87ba4",  # 5 hồng
    "#008300",  # 6 lục
    "#4a3aa7",  # 7 tím
    "#e34948",  # 8 đỏ
]

# Màu trạng thái — chỉ dùng khi màu MANG NGHĨA tốt/xấu, không bao giờ dùng làm màu chuỗi
STATUS = {"good": "#0ca30c", "warning": "#fab219", "serious": "#ec835a", "critical": "#d03b3b"}

# Thang tuần tự một sắc (nhạt → đậm) cho heatmap
SEQ = LinearSegmentedColormap.from_list(
    "seq_blue",
    ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"],
)
# Thang phân kỳ, điểm giữa là xám trung tính. Quy ước một lần cho cả notebook:
# ÂM = ĐỎ, DƯƠNG = XANH. Nhờ vậy ở mọi hình dùng thang này, xanh luôn là phía tốt/cao hơn.
DIV = LinearSegmentedColormap.from_list(
    "div_red_blue", ["#b02c2c", "#e88b8a", "#f0efec", "#6da7ec", "#184f95"]
)

mpl.rcParams.update({
    "figure.facecolor": SURFACE,
    "figure.dpi": 110,
    "savefig.dpi": 200,
    "savefig.facecolor": SURFACE,
    "savefig.bbox": "tight",
    "axes.facecolor": SURFACE,
    "axes.edgecolor": AXIS,
    "axes.linewidth": 0.8,
    "axes.labelcolor": INK_2,
    "axes.titlecolor": INK,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "axes.axisbelow": True,
    "grid.color": GRID,
    "grid.linewidth": 0.8,
    "grid.linestyle": "-",          # lưới nét liền, mảnh — nét đứt bị đọc nhầm thành "ngưỡng"
    "xtick.color": INK_2,
    "ytick.color": INK_2,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "text.color": INK,
    "font.family": "sans-serif",
    "font.sans-serif": ["Segoe UI", "Tahoma", "DejaVu Sans", "Arial"],
    "font.size": 10,
    "legend.frameon": False,
    "legend.fontsize": 9,
})


def ghi_chan(fig, note: str, y: float = -0.02):
    """Chú thích chân hình, tự xuống dòng theo bề ngang thật của hình.

    Không ngắt dòng thì bbox_inches='tight' sẽ nong cả hình ra cho vừa một dòng chữ dài.
    """
    rong = max(60, int(fig.get_figwidth() * 15))       # ~15 ký tự trên mỗi inch ở cỡ chữ 8,2
    fig.text(0.005, y, textwrap.fill(note, rong), fontsize=8.2, color=INK_3, va="top", ha="left")


def frame(ax, title: str, subtitle: str | None = None, note: str | None = None):
    """Tiêu đề 2 tầng + chú thích nguồn/bộ lọc ở chân hình."""
    ax.set_title(title, fontsize=13, fontweight="600", loc="left", pad=18 if subtitle else 10)
    if subtitle:
        ax.text(0, 1.015, subtitle, transform=ax.transAxes, fontsize=9.5,
                color=INK_2, va="bottom", ha="left")
    if note:
        ghi_chan(ax.figure, note)
    return ax


def save(fig, name: str):
    if SAVE_FIG:
        out = FIGDIR / f"{name}.png"
        fig.savefig(out)
        print("đã lưu →", out.relative_to(ROOT))


def pc(v, digits=2):
    """Định dạng phần trăm, ô trống ghi '—' chứ không ghi 0."""
    return "—" if pd.isna(v) else f"{v:.{digits}f}%"

### Hai hàm dán nhãn

Nhãn chồng lên nhau là lỗi hay gặp nhất khi có nhiều ngân hàng. Hai hàm dưới đây đẩy nhãn ra
cho tới khi hết chồng, làm việc trong toạ độ **màn hình** nên đúng với mọi kích thước hình.

In [ ]:
def _y_display(ax, y):
    """Toạ độ y trên màn hình — với trục thường, y hiển thị không phụ thuộc x."""
    return ax.transData.transform((ax.get_xlim()[0], y))[1]


def _y_data(ax, y_disp):
    return ax.transData.inverted().transform((0, y_disp))[1]


def nhan_cuoi(ax, items, dx=9, gap=15, fontsize=9):
    """Dán nhãn ở cuối đường. items = [(x, y, text, color)]; nhãn nào sát nhau thì đẩy dọc ra."""
    ax.figure.canvas.draw()
    order = sorted(range(len(items)), key=lambda i: items[i][1])
    ys = [_y_display(ax, items[i][1]) for i in order]
    for k in range(1, len(ys)):                       # đẩy lên khi quá sát
        ys[k] = max(ys[k], ys[k - 1] + gap)
    for k, i in enumerate(order):
        x, y, text, color = items[i]
        ax.annotate(text, (x, _y_data(ax, ys[k])), xytext=(dx, 0),
                    textcoords="offset points", color=color, fontsize=fontsize,
                    va="center", ha="left", fontweight="600", annotation_clip=False)


def cho_nhan(ax, labels, fontsize=9, extra=22):
    """Nới trục x vừa đủ chỗ cho nhãn cuối đường — không thừa khoảng trắng, không cắt chữ."""
    ax.figure.canvas.draw()
    w = max(len(t) for t in labels) * fontsize * 0.62 + extra
    inv = ax.transData.inverted()
    dx = inv.transform((w, 0))[0] - inv.transform((0, 0))[0]
    x0, x1 = ax.get_xlim()
    ax.set_xlim(x0, x1 + dx)


def _cham(a, b):
    return not (a[2] <= b[0] or b[2] <= a[0] or a[3] <= b[1] or b[3] <= a[1])


def nhan_diem(ax, xs, ys, texts, color=INK_2, radii=None, fontsize=8.5):
    """Dán nhãn cạnh từng điểm scatter, tự tránh chồng lên nhãn đã đặt.

    radii: bán kính điểm tính bằng point (để nhãn không đè lên bong bóng lớn).
    """
    ax.figure.canvas.draw()
    dpi = ax.figure.dpi
    da = [] if radii is None else list(radii)
    da = [0] * len(texts) if not da else da
    hop = []
    for x, y, t, r in zip(xs, ys, texts, da):
        px, py = ax.transData.transform((x, y))
        d = r * dpi / 72 + 9                          # bán kính điểm + khoảng thở
        w, h = len(t) * fontsize * 0.66, fontsize * 1.55
        for dx, dy, ha in [(0, d, "center"), (0, -d, "center"), (d, 0, "left"),
                           (-d, 0, "right"), (0, d + h, "center"), (0, -d - h, "center")]:
            cx, cy = px + dx, py + dy
            x0 = cx - w / 2 if ha == "center" else (cx if ha == "left" else cx - w)
            box = (x0, cy - h / 2, x0 + w, cy + h / 2)
            if not any(_cham(box, b) for b in hop):
                break
        hop.append(box)
        ax.annotate(t, (x, y), xytext=(dx, dy), textcoords="offset points",
                    ha=ha, va="center", fontsize=fontsize, color=color)

## 1. Nạp mô hình sao

`load()` đọc `utf-8-sig` (đúng như `export_powerbi.py` ghi ra) và parse sẵn cột ngày.
`LISTED` là 22 mã niêm yết — mọi bảng fact đều được lọc qua nó ngay từ đây.

Lưu ý đơn vị: cột `*_ty_vnd` là **tỷ đồng** (VND chia 1e9). Các biểu đồ quy mô ở dưới chia
thêm 1.000 để hiển thị bằng **nghìn tỷ** cho nhãn trục ngắn gọn.

In [ ]:
def load(name: str, dates: list[str] | None = None, **kw) -> pd.DataFrame:
    df = pd.read_csv(PBI / f"{name}.csv", encoding="utf-8-sig", **kw)
    for c in dates or []:
        df[c] = pd.to_datetime(df[c], errors="coerce")
    return df


dim_bank = load("dim_bank")
LISTED = dim_bank.loc[dim_bank["is_listed"], "bank_key"].tolist()
NAME = dim_bank.set_index("bank_key")["bank_name"].to_dict()      # bank_key → tên hiển thị
TICKER = dim_bank.set_index("bank_key")["ticker"].to_dict()
NGHIN_TY = 1_000                                                  # tỷ đồng → nghìn tỷ đồng


def listed(df: pd.DataFrame) -> pd.DataFrame:
    """Bẫy #4: bỏ 31 ngân hàng ngoài sàn + dòng UNKNOWN."""
    return df[df["bank_key"].isin(LISTED)].copy()


fact_deposit = listed(load("fact_deposit_rate", dates=["date"]))
fact_lending = listed(load("fact_lending_rate_monthly", dates=["month_date"]))
fact_quarter = listed(load("fact_bank_quarterly", dates=["period_end"]))
fact_app = listed(load("fact_app_review_monthly", dates=["month_date"]))
fact_branch = listed(load("fact_branch_by_province"))
fact_point = listed(load("fact_branch_point"))
fact_price = listed(load("fact_price_daily", dates=["date"]))
fact_fin = listed(load("fact_financial", dates=["period_end"]))
macro = load("fact_macro_monthly", dates=["month_date"])
credit_growth = load("fact_credit_growth_yearly", dates=["year_end"])
dim_metric = load("dim_metric")

print(f"{len(LISTED)} ngân hàng niêm yết: {', '.join(sorted(TICKER[b] for b in LISTED))}")
for n, d in [("deposit", fact_deposit), ("lending", fact_lending), ("quarterly", fact_quarter),
             ("financial", fact_fin), ("app", fact_app), ("branch_point", fact_point),
             ("price", fact_price)]:
    print(f"  {n:<13} {len(d):>7,} dòng")

## 2. Nhóm ngân hàng — cách phân biệt 22 mã mà mắt theo kịp

22 ngân hàng thì không thể cho mỗi ngân hàng một màu: quá 8 màu là mắt người hết phân biệt được,
nhất là với người mù màu. Notebook này giải quyết bằng **ba tầng**, dùng nhất quán ở mọi hình:

1. **Màu theo nhóm** (3 màu) — nói lên điều gì đó về ngân hàng, không chỉ để tô cho khác nhau.
   Màu gắn với ngân hàng, lọc bớt ngân hàng khác cũng không đổi màu ngân hàng còn lại.
2. **Tên viết thẳng trên hình** — trục y của biểu đồ cột, nhãn cuối đường, nhãn cạnh điểm.
   Đây mới là thứ phân biệt được từng ngân hàng; màu chỉ để gom nhóm.
3. **Lưới nhỏ (small multiples)** — mỗi ngân hàng một ô riêng có tên, nền là 21 ngân hàng còn
   lại màu xám. Xem được từng ngân hàng một mà vẫn so được với cả ngành.

Ranh giới nhóm: 3 ngân hàng gốc quốc doanh, phần tư nhân cắt theo **tổng tài sản** ở mốc
800 nghìn tỷ — mốc này rơi đúng vào chỗ trống lớn nhất trong bảng xếp hạng (STB 892 ↔ LPB 616).

In [ ]:
QUOC_DOANH = {"VCB", "CTG", "BID"}       # gốc quốc doanh, Nhà nước còn chi phối
NGUONG_LON = 800                          # nghìn tỷ tổng tài sản, ranh giới nhóm tư nhân

_ts = (fact_quarter.sort_values("period_end").groupby("bank_key")["total_assets_ty_vnd"]
       .last() / NGHIN_TY)
NHOM = {b: ("Quốc doanh" if b in QUOC_DOANH else
            "Tư nhân lớn" if _ts.get(b, 0) >= NGUONG_LON else "Tư nhân vừa & nhỏ")
        for b in LISTED}
NHOM_ORDER = ["Quốc doanh", "Tư nhân lớn", "Tư nhân vừa & nhỏ"]
MAU_NHOM = dict(zip(NHOM_ORDER, SERIES[:3]))       # 3 slot đầu — đạt chuẩn cả với mù màu


def mau(b) -> str:
    """Màu của một ngân hàng — theo nhóm, không bao giờ theo thứ hạng trong biểu đồ."""
    return MAU_NHOM[NHOM[b]]


def chu_thich_nhom(ax, **kw):
    """Chú giải 3 nhóm, dùng chung cho mọi hình có tô màu theo nhóm."""
    kw.setdefault("loc", "lower right")
    return ax.legend([Line2D([], [], color=MAU_NHOM[n], lw=6) for n in NHOM_ORDER],
                     [f"{n} ({sum(v == n for v in NHOM.values())})" for n in NHOM_ORDER], **kw)


for n in NHOM_ORDER:
    ds = [TICKER[b] for b in LISTED if NHOM[b] == n]
    print(f"{n:<20} {len(ds):>2} mã: {', '.join(sorted(ds))}")

### Lưới nhỏ — mỗi ngân hàng một ô

`luoi_nho()` vẽ 22 ô, mỗi ô một ngân hàng: đường của chính ngân hàng đó tô màu theo nhóm, 21
đường còn lại để xám mảnh làm nền so sánh. Mọi ô dùng chung thang đo nên so ngang được, và
**không ô nào phải đoán xem đường nào là của ai**.

In [ ]:
def khung_y(v: pd.Series):
    """Khung y bền với ngoại lai: chỉ cắt khi một vài điểm cực đoan bóp dẹt phần còn lại."""
    v = pd.to_numeric(v, errors="coerce").dropna()
    if v.empty:
        return None, 0
    lo_r, hi_r = np.percentile(v, [1, 99])
    if hi_r - lo_r <= 0 or (v.max() - v.min()) <= 2.5 * (hi_r - lo_r):
        return None, 0                                   # phân bố lành, để matplotlib tự co
    pad = (hi_r - lo_r) * 0.15
    lo_r, hi_r = lo_r - pad, hi_r + pad
    return (lo_r, hi_r), int(((v < lo_r) | (v > hi_r)).sum())


def luoi_nho(df, cot_x, cot_y, tieu_de, phu_de, ghi_chu, ten_file,
             ncols=5, sap_xep=None, yfmt=None, cao_o=1.55, kieu="line", moc_y=None):
    """Small multiples: mỗi ngân hàng một ô, nền là toàn bộ các ngân hàng khác."""
    banks = list(sap_xep) if sap_xep is not None else sorted(
        df["bank_key"].unique(), key=lambda b: -_ts.get(b, 0))
    nrows = -(-len(banks) // ncols)
    ylim, n_ngoai = khung_y(df[cot_y])
    if n_ngoai:
        ghi_chu += (f" {n_ngoai} điểm ngoại lai nằm ngoài khung đã bị cắt để 22 ô còn đọc được — "
                    f"ô nào bị cắt đều ghi rõ giá trị thật.")
    fig, axes = plt.subplots(nrows, ncols, figsize=(2.5 * ncols, cao_o * nrows),
                             sharex=True, sharey=True)
    axes = np.atleast_1d(axes).ravel()
    nen = [(g[cot_x].values, g[cot_y].values) for _, g in df.groupby("bank_key")]

    for ax, b in zip(axes, banks):
        if moc_y is not None:
            ax.axhline(moc_y, color=INK_3, lw=0.8, zorder=2)
        for gx, gy in nen:
            ax.plot(gx, gy, color=MUTED, lw=0.7, alpha=0.55, zorder=1)
        g = df[df["bank_key"] == b].sort_values(cot_x)
        c = mau(b)
        if kieu == "marker":
            ax.plot(g[cot_x], g[cot_y], color=c, lw=1.6, marker="o", ms=2.6,
                    mec=SURFACE, mew=0.6, zorder=3)
        else:
            ax.plot(g[cot_x], g[cot_y], color=c, lw=1.8, zorder=3)
        cuoi = g[cot_y].dropna()
        nhan = f"{NAME[b]}" + (f" · {cuoi.iloc[-1]:.2f}" if len(cuoi) else "")
        ax.set_title(nhan, fontsize=9, color=c, fontweight="600", loc="left", pad=4)
        if ylim is not None:
            ax.set_ylim(*ylim)
            for v, ha_y in [(cuoi.min(), 0.04), (cuoi.max(), 0.9)]:      # ghi giá trị bị cắt
                if len(cuoi) and not (ylim[0] <= v <= ylim[1]):
                    ax.text(0.97, ha_y, f"{'↓' if v < ylim[0] else '↑'} {v:,.1f}",
                            transform=ax.transAxes, ha="right", fontsize=7.5, color=c,
                            fontweight="600")
        ax.tick_params(labelsize=7.5)
        ax.grid(lw=0.6)
        if yfmt:
            ax.yaxis.set_major_formatter(yfmt)
        if np.issubdtype(np.asarray(df[cot_x]).dtype, np.datetime64):
            loc = mdates.AutoDateLocator(minticks=2, maxticks=4)
            ax.xaxis.set_major_locator(loc)
            ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(loc))
    for ax in axes[len(banks):]:
        ax.set_visible(False)
    for ax in axes[:len(banks)]:
        ax.label_outer()

    # Chừa chỗ cho tiêu đề theo chiều cao thật của hình, để 2 dòng không chồng nhau
    cao = fig.get_figheight()
    fig.tight_layout(rect=[0, 0, 1, 1 - 0.62 / cao])
    fig.text(0.005, 1 - 0.06 / cao, tieu_de, fontsize=13, fontweight="600", ha="left", va="top")
    fig.text(0.005, 1 - 0.32 / cao, phu_de, fontsize=9.5, color=INK_2, ha="left", va="top")
    ghi_chan(fig, ghi_chu, y=-0.004)
    save(fig, ten_file)
    plt.show()

## 3. Độ đầy dữ liệu — quyết định *được vẽ cái gì*

README cảnh báo: `car_percent` chỉ có 11,8% số dòng, vẽ ra sẽ trông như các ngân hàng khác có
CAR bằng 0. Nên trước khi vẽ bất cứ thứ gì, xem độ đầy của từng cột trong `fact_bank_quarterly`.
Ngưỡng 70% là mức mà một biểu đồ đường theo quý còn liền mạch.

In [ ]:
COVER_COLS = [
    "loans_ty_vnd", "deposits_ty_vnd", "total_assets_ty_vnd", "ldr_computed_percent",
    "loans_qoq_percent", "loans_yoy_percent", "ldr_percent", "nim_percent",
    "npl_percent", "npl_coverage_percent", "cof_ttm_percent",
    "earning_asset_yield_ttm_percent", "nim_ttm_percent", "car_percent",
]
LABEL_VN = {
    "loans_ty_vnd": "Dư nợ cho vay", "deposits_ty_vnd": "Tiền gửi",
    "total_assets_ty_vnd": "Tổng tài sản", "ldr_computed_percent": "LDR (tự tính)",
    "loans_qoq_percent": "Tăng trưởng dư nợ QoQ", "loans_yoy_percent": "Tăng trưởng dư nợ YoY",
    "ldr_percent": "LDR (BCTC)", "nim_percent": "NIM quý", "npl_percent": "Tỷ lệ nợ xấu (NPL)",
    "npl_coverage_percent": "Tỷ lệ bao phủ nợ xấu", "cof_ttm_percent": "Chi phí vốn (COF, TTM)",
    "earning_asset_yield_ttm_percent": "Lợi suất tài sản sinh lời (TTM)",
    "nim_ttm_percent": "NIM (TTM)", "car_percent": "Hệ số an toàn vốn (CAR)",
}

cov = (fact_quarter[COVER_COLS].notna().mean() * 100).sort_values()
THRESH = 70

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh([LABEL_VN[c] for c in cov.index], cov.values, height=0.68,
        color=[SERIES[0] if v >= THRESH else MUTED for v in cov])
ax.axvline(THRESH, color=INK_3, lw=1)
ax.text(THRESH + 1.5, len(cov) - 0.45, f"ngưỡng {THRESH}%", color=INK_3, fontsize=8.5, va="center")
for y, v in enumerate(cov.values):
    ax.text(v + 1, y, f"{v:.1f}%", va="center", fontsize=8.5,
            color=INK_2 if v >= THRESH else INK_3)
ax.set_xlim(0, 112)
ax.set_xlabel("% số dòng có giá trị")
ax.xaxis.set_major_formatter(lambda v, p: f"{v:.0f}%")
ax.grid(axis="y", visible=False)
frame(ax, "Cột nào của fact_bank_quarterly đủ đầy để vẽ?",
      f"748 dòng · 22 ngân hàng niêm yết · 2018-Q1 → {fact_quarter['period'].max()}",
      "Xám = dưới ngưỡng, chỉ nên dùng trong bảng chứ không nên vẽ. CAR 11,8% là ví dụ trong README: "
      "vẽ lên sẽ trông như phần lớn ngân hàng có CAR = 0.")
save(fig, "fig_00_do_day_du_lieu")
plt.show()

## 4. Bối cảnh vĩ mô

### 4.1 Lãi suất điều hành và CPI

README gợi ý "line chart 2 trục" — ở đây **tách thành hai panel dùng chung trục thời gian**.
Hai trục y trên cùng một khung khiến vị trí giao nhau của hai đường trở thành ngẫu nhiên
(phụ thuộc cách canh thang đo), người đọc lại hiểu thành mối tương quan có thật.

In [ ]:
POLICY = {
    "policy_tai_cap_von_percent": "Tái cấp vốn",
    "policy_tai_chiet_khau_percent": "Tái chiết khấu",
    "policy_cho_vay_qua_dem_percent": "Cho vay qua đêm",
}
m = macro.dropna(subset=list(POLICY), how="all")

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7), sharex=True,
                               gridspec_kw={"height_ratios": [1.35, 1], "hspace": 0.28})

cuoi = []
for i, (col, lab) in enumerate(POLICY.items()):
    ax1.plot(m["month_date"], m[col], color=SERIES[i], lw=2, drawstyle="steps-post", label=lab)
    r = m.dropna(subset=[col]).iloc[-1]
    cuoi.append((r["month_date"], r[col], f"{lab} {r[col]:.2f}%", SERIES[i]))
ax1.set_ylabel("% / năm")
ax1.legend(loc="upper right", ncol=3)
ax1.yaxis.set_major_formatter(lambda v, p: f"{v:.0f}%")
ax1.set_xlim(m["month_date"].min(), m["month_date"].max() + pd.Timedelta(days=30))
frame(ax1, "Lãi suất điều hành của NHNN và lạm phát",
      "Lãi suất điều hành trải theo tháng (forward-fill từ 33 sự kiện) · CPI so cùng kỳ")

cpi = macro.dropna(subset=["cpi_yoy_percent"])
ax2.plot(cpi["month_date"], cpi["cpi_yoy_percent"], color=SERIES[3], lw=2)
ax2.fill_between(cpi["month_date"], 0, cpi["cpi_yoy_percent"], color=SERIES[3], alpha=0.10)
ax2.axhline(4, color=INK_3, lw=1)
ax2.text(0.34, 4.2, "mục tiêu 4%", transform=ax2.get_yaxis_transform(),
         color=INK_3, fontsize=8.5, va="bottom")
r = cpi.iloc[-1]
ax2.set_ylabel("CPI so cùng kỳ (%)")
ax2.yaxis.set_major_formatter(lambda v, p: f"{v:.0f}%")
nhan_cpi = [(r["month_date"], r["cpi_yoy_percent"], f"CPI {r['cpi_yoy_percent']:.2f}%", SERIES[3])]
cho_nhan(ax2, [it[2] for it in cuoi + nhan_cpi])        # hai panel dùng chung trục x
nhan_cuoi(ax1, cuoi)
nhan_cuoi(ax2, nhan_cpi)
ghi_chan(fig, "Nguồn: NHNN (policy_rate) + NSO/VnEconomy (CPI) qua fact_macro_monthly. "
              "Hai đơn vị khác nhau → hai panel, không dùng hai trục y.", y=0.02)
save(fig, "fig_01_lai_suat_dieu_hanh_cpi")
plt.show()

### 4.2 Tăng trưởng tín dụng toàn hệ thống

In [ ]:
cg = credit_growth.sort_values("year")
fig, ax = plt.subplots(figsize=(8.5, 4.2))
ax.bar(cg["year"].astype(str), cg["credit_growth_percent"], color=SERIES[0], width=0.62)
for x, v in zip(cg["year"].astype(str), cg["credit_growth_percent"]):
    ax.text(x, v + 0.35, f"{v:.2f}%", ha="center", fontsize=9, color=INK_2)
tb = cg["credit_growth_percent"].mean()
ax.axhline(tb, color=INK_3, lw=1)
ax.text(len(cg) - 0.4, tb + 0.25, f"TB {tb:.1f}%", color=INK_3, fontsize=8.5, ha="right")
ax.set_ylim(0, cg["credit_growth_percent"].max() * 1.18)
ax.set_ylabel("%")
ax.yaxis.set_major_formatter(lambda v, p: f"{v:.0f}%")
ax.grid(axis="x", visible=False)
frame(ax, "Tăng trưởng tín dụng toàn hệ thống theo năm",
      "NHNN công bố · fact_credit_growth_yearly",
      "Chỉ 8 năm số liệu nên vẽ cột; vẽ đường sẽ gợi ý một xu hướng liên tục không có trong dữ liệu.")
save(fig, "fig_02_tang_truong_tin_dung")
plt.show()

## 5. Gửi tiết kiệm — câu hỏi trung tâm của khóa luận

**Bẫy #1.** `fact_deposit_rate` chồng ba nguồn đo *cùng một ô* và lệch nhau tới 2,1 điểm.
Kéo `rate_percent` ra mà không lọc thì được trung bình của 3 nguồn × 9 kỳ hạn × 2 kênh —
một con số vô nghĩa nhưng biểu đồ vẫn vẽ đẹp. Ba tham số dưới đây đóng vai trò "slicer".

Chọn `24hmoney` vì đủ **22/22** mã và lùi được về 2023 (`laisuattietkiem` chỉ 17/22 mã, riêng
ô 12 tháng/online chỉ có 12 mã; `cafef` có 58 ngày nhưng chỉ đúng ô 12 tháng).

In [ ]:
DEP_SOURCE = "24hmoney"      # 24hmoney | laisuattietkiem | cafef
DEP_TERM = 12                # kỳ hạn, tháng
DEP_CHANNEL = "online"       # online | quay

dep = fact_deposit[fact_deposit["source"] == DEP_SOURCE]
SNAP = dep["date"].max()                       # ngày chụp gần nhất của nguồn đó
SNAP_TXT = SNAP.strftime("%d/%m/%Y")

snap = dep[(dep["date"] == SNAP) & (dep["term_months"] == DEP_TERM)
           & (dep["channel"] == DEP_CHANNEL)]
rank = (snap.groupby("bank_key")["rate_percent"].mean()
        .rename("rate").reset_index().sort_values("rate"))
rank["ten"] = rank["bank_key"].map(NAME)

print(f"Nguồn {DEP_SOURCE} · {DEP_TERM} tháng · {DEP_CHANNEL} · ngày {SNAP_TXT} "
      f"· {len(rank)}/{len(LISTED)} ngân hàng")

### 5.1 Ngân hàng nào trả lãi cao nhất?

In [ ]:
def dai_nganh(ax, cot_min: str, cot_max: str, nhan: str):
    """Tô dải bình quân ngành do NHNN/báo cáo thị trường công bố, tháng gần nhất có số liệu."""
    b = macro.dropna(subset=[cot_min, cot_max])
    if b.empty:
        return None
    r = b.iloc[-1]
    ax.axvspan(r[cot_min], r[cot_max], color=INK_3, alpha=0.09, zorder=0)
    ax.annotate(f"{nhan} {r[cot_min]:.1f}–{r[cot_max]:.1f}% ({r['month']})",
                xy=((r[cot_min] + r[cot_max]) / 2, 1.0), xycoords=ax.get_xaxis_transform(),
                xytext=(0, 6), textcoords="offset points",
                ha="center", va="bottom", fontsize=8.2, color=INK_3)
    return r


fig, ax = plt.subplots(figsize=(9.5, 7))
dai_nganh(ax, "deposit_12_to_24m_min", "deposit_12_to_24m_max", "dải bình quân ngành")
ax.barh(rank["ten"], rank["rate"], color=[mau(b) for b in rank["bank_key"]], height=0.66)
for y, v in enumerate(rank["rate"]):
    ax.text(v + 0.06, y, f"{v:.2f}%", va="center", fontsize=9, color=INK_2)
med = rank["rate"].median()
ax.axvline(med, color=INK_3, lw=1)
ax.text(med + 0.05, len(rank) - 0.4, f"trung vị {med:.2f}%", color=INK_3, fontsize=8.5, va="center")
ax.set_xlim(0, rank["rate"].max() * 1.14)
ax.set_xlabel("Lãi suất niêm yết (%/năm)")
ax.xaxis.set_major_formatter(lambda v, p: f"{v:.0f}%")
ax.grid(axis="y", visible=False)
chu_thich_nhom(ax, loc="lower right")
frame(ax, f"Xếp hạng lãi suất tiết kiệm {DEP_TERM} tháng, kênh {DEP_CHANNEL}",
      f"Ảnh chụp ngày {SNAP_TXT} · nguồn {DEP_SOURCE} · {len(rank)}/22 ngân hàng niêm yết",
      "Tên ngân hàng nằm ngay trên trục nên phân biệt được từng mã; màu chỉ để thấy nhóm nào "
      "đang nằm ở đâu. Vệt xám dọc là dải lãi suất 12–24 tháng bình quân toàn ngành — cột vượt ra "
      "ngoài vệt là ngân hàng trả cao/thấp hơn mặt bằng. Đổi nguồn thì con số sẽ khác: ba nguồn đo "
      "cùng một ô lệch nhau tới 2,1 điểm.")
save(fig, "fig_03_xep_hang_lai_suat_tien_gui")
plt.show()

### 5.2 Cao nhất ở kỳ hạn nào? — ma trận ngân hàng × kỳ hạn

Độ lớn liên tục nên dùng thang **một sắc, nhạt → đậm**. Con số in ngay trong ô đóng luôn vai trò
bảng tra — không ai phải đoán giá trị từ sắc độ.

In [ ]:
mat = (dep[(dep["date"] == SNAP) & (dep["channel"] == DEP_CHANNEL)]
       .pivot_table(index="bank_key", columns="term_months", values="rate_percent", aggfunc="mean")
       .reindex(rank.sort_values("rate", ascending=False)["bank_key"]))
mat.index = [NAME[b] for b in mat.index]

fig, ax = plt.subplots(figsize=(9.5, 7.5))
lo, hi = np.nanmin(mat.values), np.nanmax(mat.values)
im = ax.imshow(mat.values, cmap=SEQ, aspect="auto", vmin=lo, vmax=hi)
ax.set_xticks(range(mat.shape[1]), [f"{c} tháng" for c in mat.columns])
ax.set_yticks(range(mat.shape[0]), mat.index)
ax.set_xticks(np.arange(-0.5, mat.shape[1], 1), minor=True)
ax.set_yticks(np.arange(-0.5, mat.shape[0], 1), minor=True)
ax.grid(which="minor", color=SURFACE, lw=2)     # khe hở giữa các ô, không phải viền quanh ô
ax.grid(which="major", visible=False)
ax.tick_params(which="minor", length=0)
for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        v = mat.values[i, j]
        if np.isnan(v):
            ax.text(j, i, "—", ha="center", va="center", fontsize=8.5, color=INK_3)
        else:
            dam = (v - lo) / (hi - lo) > 0.55
            ax.text(j, i, f"{v:.1f}", ha="center", va="center", fontsize=8.5,
                    color="#ffffff" if dam else INK)
cb = fig.colorbar(im, ax=ax, pad=0.02, fraction=0.03)
cb.set_label("%/năm", color=INK_2, fontsize=9)
cb.outline.set_visible(False)
frame(ax, "Lãi suất tiết kiệm theo kỳ hạn",
      f"Ảnh chụp {SNAP_TXT} · nguồn {DEP_SOURCE} · kênh {DEP_CHANNEL} · sắp theo kỳ hạn {DEP_TERM} tháng",
      "Ô '—' là kỳ hạn ngân hàng đó không niêm yết ở nguồn này, không phải lãi suất bằng 0.")
save(fig, "fig_04_ma_tran_ky_han")
plt.show()

# bảng tra tương ứng với hình trên
mat.round(2)

### 5.3 Diễn biến theo thời gian

**Độ phủ quyết định loại biểu đồ.** `24hmoney` chỉ có **27 ngày** rải từ 2023-01 đến 2026-09
(ảnh chụp qua web.archive.org, không phải cào hàng ngày). Trục X phải để **liên tục theo ngày
thật** và chấm điểm tại đúng ngày có số liệu — xếp đều 27 mốc cạnh nhau sẽ trông như dữ liệu
hàng ngày.

22 đường 22 màu thì không ai đọc được. Ở đây tô màu **3 ngân hàng đáng chú ý**, phần còn lại để
xám mờ làm nền: vẫn thấy vùng phân bố mà không mất trọng tâm.

In [ ]:
ts = (dep[(dep["term_months"] == DEP_TERM) & (dep["channel"] == DEP_CHANNEL)]
      .groupby(["bank_key", "date"])["rate_percent"].mean().reset_index())

# Chỉ nhấn mạnh ngân hàng có chuỗi đủ dài, nếu không sẽ tô màu một đoạn cụt
du_dai = ts.groupby("bank_key")["date"].count()
ung_vien = rank[rank["bank_key"].isin(du_dai[du_dai >= du_dai.max() * 0.6].index)]
HL = dict(zip([ung_vien.iloc[-1]["bank_key"], ung_vien.iloc[0]["bank_key"],
               "VCB" if "VCB" in LISTED else ung_vien.iloc[len(ung_vien) // 2]["bank_key"]],
              SERIES[:3]))

fig, ax = plt.subplots(figsize=(10.5, 5.6))
for b, g in ts.groupby("bank_key"):
    if b not in HL:
        ax.plot(g["date"], g["rate_percent"], color=MUTED, lw=1, alpha=0.75, zorder=1)
cuoi = []
for b, c in HL.items():
    g = ts[ts["bank_key"] == b].sort_values("date")
    ax.plot(g["date"], g["rate_percent"], color=c, lw=2, zorder=3,
            marker="o", ms=4.5, mec=SURFACE, mew=1.2)
    r = g.iloc[-1]
    cuoi.append((r["date"], r["rate_percent"], f"{NAME[b]} {r['rate_percent']:.2f}%", c))
ax.legend([Line2D([], [], color=c, lw=2) for c in HL.values()] + [Line2D([], [], color=MUTED, lw=1)],
          [NAME[b] for b in HL] + [f"{ts['bank_key'].nunique() - len(HL)} ngân hàng còn lại"],
          loc="upper right", ncol=4)
ax.set_ylabel("%/năm")
ax.yaxis.set_major_formatter(lambda v, p: f"{v:.0f}%")
ax.set_xlim(ts["date"].min() - pd.Timedelta(days=45), ts["date"].max())
frame(ax, f"Lãi suất tiết kiệm {DEP_TERM} tháng ({DEP_CHANNEL}) qua các kỳ chụp",
      f"Nguồn {DEP_SOURCE} · {ts['date'].nunique()} ngày chụp, "
      f"{ts['date'].min():%m/%Y} → {ts['date'].max():%m/%Y}",
      "Mỗi chấm là một ngày thực sự có số liệu; khoảng trắng giữa hai chấm là khoảng không quan sát, "
      "không phải lãi suất đi ngang.")
cho_nhan(ax, [it[2] for it in cuoi])
nhan_cuoi(ax, cuoi)
save(fig, "fig_05_dien_bien_lai_suat_tien_gui")
plt.show()

Hình trên trả lời "cả ngành đi về đâu", nhưng chỉ đọc được 3 ngân hàng. Muốn xem **từng ngân
hàng một** thì dùng lưới nhỏ: 22 ô, mỗi ô một ngân hàng có tên và giá trị mới nhất, nền xám là
21 ngân hàng còn lại. Ô sắp theo tổng tài sản giảm dần.

In [ ]:
luoi_nho(ts, "date", "rate_percent",
         f"Lãi suất tiết kiệm {DEP_TERM} tháng ({DEP_CHANNEL}) — từng ngân hàng",
         f"Nguồn {DEP_SOURCE} · {ts['date'].nunique()} ngày chụp · ô sắp theo tổng tài sản · "
         f"số cạnh tên là lãi suất ở kỳ chụp gần nhất",
         "Mọi ô dùng chung một thang đo nên so ngang được. Nền xám là 21 ngân hàng còn lại — "
         "thấy ngay ngân hàng đang nằm trên hay dưới mặt bằng chung.",
         "fig_05b_luoi_lai_suat_tien_gui",
         yfmt=lambda v, p: f"{v:.0f}%", kieu="marker")

### 5.4 Gửi online lợi hơn ra quầy bao nhiêu?

Số liệu có hai cực (âm/dương) nên đây là chỗ hợp lệ để dùng **thang phân kỳ**: đỏ ↔ xanh,
điểm giữa là xám trung tính = "không chênh".

In [ ]:
ch = (dep[(dep["date"] == SNAP) & (dep["term_months"] == DEP_TERM)]
      .pivot_table(index="bank_key", columns="channel", values="rate_percent", aggfunc="mean")
      .dropna())
ch["gap"] = ch["online"] - ch["quay"]
ch = ch.sort_values("gap")
lim = max(abs(ch["gap"].min()), abs(ch["gap"].max())) or 1.0

fig, ax = plt.subplots(figsize=(8.5, 6.6))
ax.barh([NAME[b] for b in ch.index], ch["gap"], height=0.66,
        color=[DIV(0.5 + 0.5 * (v / lim)) for v in ch["gap"]])
ax.axvline(0, color=INK_2, lw=1)
for y, v in enumerate(ch["gap"]):
    ax.text(v + (0.03 if v >= 0 else -0.03), y, f"{v:+.2f}", va="center", fontsize=8.5,
            color=INK_2, ha="left" if v >= 0 else "right")
ax.set_xlim(-lim * 1.45, lim * 1.35)
ax.set_xlabel("Chênh lệch online − quầy (điểm %)")
ax.grid(axis="y", visible=False)
frame(ax, "Gửi online được cộng thêm bao nhiêu?",
      f"Kỳ hạn {DEP_TERM} tháng · ảnh chụp {SNAP_TXT} · nguồn {DEP_SOURCE}",
      "Đỏ = ra quầy còn cao hơn online. Đây là chênh lệch niêm yết, chưa tính khuyến mại theo số tiền gửi.")
save(fig, "fig_06_online_vs_quay")
plt.show()

## 6. Vay — ở đâu rẻ nhất?

**Độ đầy rất lệch.** Từ Công điện 18/CĐ-TTg mỗi ngân hàng phải tự công bố lãi suất cho vay bình
quân hàng tháng, nhưng mỗi ngân hàng chỉ có **1–15 tháng** (VPB đúng 1 tháng, BID/ACB/LPB có 15).
Vẽ đường thì phần lớn ngân hàng thành một chấm rời — nên phần này dùng **cột**, và mỗi cột
**kèm luôn tháng công bố** để người đọc biết các cột không cùng một mốc thời gian.

In [ ]:
lend_last = (fact_lending.dropna(subset=["lending_avg_percent"])
             .sort_values("month_date").groupby("bank_key").tail(1)
             .sort_values("lending_avg_percent", ascending=False))   # rẻ nhất nằm trên cùng
so_thang = fact_lending.groupby("bank_key")["month"].nunique()

fig, ax = plt.subplots(figsize=(10, 7))
dai_nganh(ax, "loan_vnd_avg_min", "loan_vnd_avg_max", "dải cho vay bình quân ngành")
ax.barh([NAME[b] for b in lend_last["bank_key"]], lend_last["lending_avg_percent"],
        color=[mau(b) for b in lend_last["bank_key"]], height=0.66)
for y, (_, r) in enumerate(lend_last.iterrows()):
    ax.text(r["lending_avg_percent"] + 0.08, y,
            f"{r['lending_avg_percent']:.2f}%   ({r['month']}, {so_thang[r['bank_key']]} tháng số liệu)",
            va="center", fontsize=8.5, color=INK_2)
ax.set_xlim(0, lend_last["lending_avg_percent"].max() * 1.40)
ax.set_xlabel("Lãi suất cho vay bình quân đã công bố (%/năm)")
ax.xaxis.set_major_formatter(lambda v, p: f"{v:.0f}%")
ax.grid(axis="y", visible=False)
chu_thich_nhom(ax, loc="lower right")
frame(ax, "Vay ở đâu rẻ nhất?",
      f"Bản công bố gần nhất của từng ngân hàng · {fact_lending['month'].min()} → {fact_lending['month'].max()}",
      "Mỗi cột là tháng công bố gần nhất CỦA CHÍNH ngân hàng đó, nên các cột không cùng mốc thời gian — "
      "tháng ghi ngay cạnh giá trị. Vệt xám dọc là dải cho vay bình quân toàn ngành. Đây là bình quân "
      "toàn danh mục, không phải lãi suất một khoản vay cụ thể.")
save(fig, "fig_07_lai_suat_cho_vay")
plt.show()

### 6.1 Ngân hàng nào ăn chênh lệch dày nhất?

`spread_vs_deposit_point` = lãi suất cho vay bình quân − lãi suất huy động bình quân, tại cùng
bản công bố.

In [ ]:
sc = lend_last.dropna(subset=["spread_vs_deposit_point"])

fig, ax = plt.subplots(figsize=(9.5, 6.6))
ax.scatter(sc["lending_avg_percent"], sc["spread_vs_deposit_point"], s=90,
           color=[mau(b) for b in sc["bank_key"]], edgecolor=SURFACE, linewidth=2, zorder=3)
mx, my = sc["lending_avg_percent"].median(), sc["spread_vs_deposit_point"].median()
ax.axvline(mx, color=INK_3, lw=1)
ax.axhline(my, color=INK_3, lw=1)
ax.margins(x=0.09, y=0.12)
ax.text(mx + 0.04, ax.get_ylim()[0], f"trung vị {mx:.2f}%", color=INK_3, fontsize=8.5, va="bottom")
ax.text(0.62, my + 0.05, f"trung vị {my:.2f} điểm", transform=ax.get_yaxis_transform(),
        color=INK_3, fontsize=8.5, va="bottom")
nhan_diem(ax, sc["lending_avg_percent"], sc["spread_vs_deposit_point"],
          [TICKER[b] for b in sc["bank_key"]], radii=[np.sqrt(90 / np.pi)] * len(sc))
ax.set_xlabel("Lãi suất cho vay bình quân (%/năm)")
ax.set_ylabel("Chênh lệch cho vay − huy động (điểm %)")
chu_thich_nhom(ax, loc="lower right")
frame(ax, "Cho vay đắt có đi cùng chênh lệch dày?",
      "Bản công bố gần nhất của từng ngân hàng · chỉ các ngân hàng có đủ cả hai vế",
      "Góc trên phải = vay đắt và chênh lệch dày. Chênh lệch ở đây tính trên số liệu ngân hàng tự công bố, "
      "không phải NIM kế toán.")
save(fig, "fig_08_chenh_lech_cho_vay")
plt.show()

### 6.2 Chuỗi dài phía cho vay — lợi suất tài sản sinh lời

Lãi suất cho vay công bố chỉ có từ 2025-01 và rất thưa, **không đủ để dự báo chuỗi thời gian**.
Chuỗi dài thay thế là `earning_asset_yield_ttm_percent` (suy ra từ NIM + COF, lùi tới 2018-Q1).
Lưu ý: đây **không phải** lãi suất cho vay — nó là lợi suất bình quân của toàn bộ tài sản sinh lời.

In [ ]:
ey = fact_quarter.dropna(subset=["earning_asset_yield_ttm_percent"])
ey_last = ey.sort_values("period_end").groupby("bank_key").tail(1).set_index("bank_key")
HL2 = dict(zip(ey_last["earning_asset_yield_ttm_percent"].nlargest(2).index.tolist()
               + ey_last["earning_asset_yield_ttm_percent"].nsmallest(1).index.tolist(), SERIES[:3]))

fig, ax = plt.subplots(figsize=(10.5, 5.6))
for b, g in ey.groupby("bank_key"):
    if b not in HL2:
        g = g.sort_values("period_end")
        ax.plot(g["period_end"], g["earning_asset_yield_ttm_percent"], color=MUTED, lw=1,
                alpha=0.8, zorder=1)
cuoi = []
for b, c in HL2.items():
    g = ey[ey["bank_key"] == b].sort_values("period_end")
    ax.plot(g["period_end"], g["earning_asset_yield_ttm_percent"], color=c, lw=2, zorder=3)
    r = g.iloc[-1]
    cuoi.append((r["period_end"], r["earning_asset_yield_ttm_percent"],
                 f"{NAME[b]} {r['earning_asset_yield_ttm_percent']:.2f}%", c))
ax.legend([Line2D([], [], color=c, lw=2) for c in HL2.values()] + [Line2D([], [], color=MUTED, lw=1)],
          [NAME[b] for b in HL2] + [f"{ey['bank_key'].nunique() - len(HL2)} ngân hàng còn lại"],
          loc="upper right", ncol=4)
ax.set_ylabel("%/năm (TTM)")
ax.yaxis.set_major_formatter(lambda v, p: f"{v:.0f}%")
ax.set_xlim(ey["period_end"].min() - pd.Timedelta(days=30), ey["period_end"].max())
frame(ax, "Lợi suất tài sản sinh lời theo quý",
      f"{ey['period'].min()} → {ey['period'].max()} · TTM 4 quý · {ey['bank_key'].nunique()} ngân hàng có số liệu",
      "Chỉ tiêu SUY RA từ NIM + chi phí vốn, độ đầy 72,2% — dùng làm chuỗi dài thay cho lãi suất cho vay "
      "công bố, nhưng không được ghi nhãn là 'lãi suất cho vay' trong luận văn.")
cho_nhan(ax, [it[2] for it in cuoi])
nhan_cuoi(ax, cuoi)
save(fig, "fig_09_loi_suat_tai_san")
plt.show()

In [ ]:
luoi_nho(ey.sort_values("period_end"), "period_end", "earning_asset_yield_ttm_percent",
         "Lợi suất tài sản sinh lời — từng ngân hàng",
         f"{ey['period'].min()} → {ey['period'].max()} · TTM 4 quý · ô sắp theo tổng tài sản",
         "Ô nào đường màu bị cụt là quý đó ngân hàng thiếu số liệu để suy ra chỉ tiêu — "
         "độ đầy chung của cột này là 72,2%.",
         "fig_09b_luoi_loi_suat_tai_san", yfmt=lambda v, p: f"{v:.0f}%")

### 6.3 Chi phí vốn so với lợi suất tài sản — một ngân hàng

Ba chuỗi cùng đơn vị (điểm %) nên nằm chung **một trục**. Đổi `BANK` để xem ngân hàng khác.

Trục hoành giữ **đủ mọi quý** trong khoảng, kể cả quý không có số liệu: bỏ quý trống đi thì
2019-Q2 sẽ đứng ngay cạnh 2022-Q2 và người đọc tưởng đó là hai quý liền nhau.

Chênh lệch ở đây là **chênh lệch gộp, KHÔNG phải NIM**: quan hệ thật là
`lợi suất = NIM + COF × (nợ phải trả lãi / tài sản sinh lời)`, nên hiệu này luôn thấp hơn NIM
bình quân khoảng 0,44 điểm. Cần NIM thì dùng thẳng cột `nim_ttm_percent`.

In [ ]:
BANK = "ACB"


def ve_cof_yield(bank_key: str):
    g = (fact_quarter[fact_quarter["bank_key"] == bank_key]
         .sort_values("period_end").set_index("period"))
    co = g["cof_ttm_percent"]
    lo_ = g["earning_asset_yield_ttm_percent"]
    chenh = lo_ - co
    x = np.arange(len(g))
    w = 0.38

    fig, ax = plt.subplots(figsize=(11, 5.2))
    ax.bar(x - w / 2 - 0.01, co, width=w, color=SERIES[0], label="Chi phí vốn (COF)")
    ax.bar(x + w / 2 + 0.01, lo_, width=w, color=SERIES[1], label="Lợi suất tài sản sinh lời")
    ax.plot(x, chenh, color=SERIES[2], lw=2, marker="o", ms=5, mec=SURFACE, mew=1.5,
            label="Chênh lệch gộp", zorder=4)
    cuoi_ = chenh.dropna()
    ax.annotate(f"{cuoi_.iloc[-1]:.2f} điểm", (x[list(g.index).index(cuoi_.index[-1])], cuoi_.iloc[-1]),
                xytext=(8, 0), textcoords="offset points", color=SERIES[2], fontsize=9,
                va="center", fontweight="600")
    thieu = int(co.isna().sum())
    ax.set_xticks(x, g.index, rotation=90, fontsize=8)
    ax.set_ylabel("%/năm (TTM)")
    ax.yaxis.set_major_formatter(lambda v, p: f"{v:.0f}%")
    ax.set_xlim(-0.8, len(g) + 1.2)
    ax.grid(axis="x", visible=False)
    ax.legend(loc="upper right", ncol=3)
    frame(ax, f"{NAME[bank_key]} — chi phí vốn, lợi suất tài sản và chênh lệch",
          f"{g.index[0]} → {g.index[-1]} · cùng đơn vị điểm % nên chung một trục"
          + (f" · {thieu} quý không có số liệu, để trống" if thieu else ""),
          "Không xếp chồng cột: COF không cộng với lợi suất, chồng lên nhau sẽ ra một tổng vô nghĩa. "
          "Đường là hiệu của hai cột — chênh lệch gộp, không phải NIM.")
    save(fig, f"fig_10_cof_vs_yield_{bank_key}")
    plt.show()


ve_cof_yield(BANK)

## 7. Sức khỏe ngân hàng

Lãi suất cao nhất không đồng nghĩa với gửi ở đó an toàn nhất. Hai hình dưới đây là vế "rủi ro"
của bài toán xếp hạng.

### 7.1 Nợ xấu và biên lãi thuần

In [ ]:
Q = fact_quarter["period"].max()
qq = fact_quarter[fact_quarter["period"] == Q].dropna(
    subset=["npl_percent", "nim_ttm_percent", "loans_ty_vnd"])
qq = qq.assign(du_no=qq["loans_ty_vnd"] / NGHIN_TY)          # tỷ đồng → nghìn tỷ đồng

fig, ax = plt.subplots(figsize=(10.5, 6.6))
S_MAX = 900
sizes = qq["du_no"] / qq["du_no"].max() * S_MAX + 40
ax.scatter(qq["npl_percent"], qq["nim_ttm_percent"], s=sizes,
           color=[mau(b) for b in qq["bank_key"]],
           alpha=0.85, edgecolor=SURFACE, linewidth=2, zorder=3)
mx, my = qq["npl_percent"].median(), qq["nim_ttm_percent"].median()
ax.axvline(mx, color=INK_3, lw=1)
ax.axhline(my, color=INK_3, lw=1)
ax.margins(x=0.10, y=0.16)
ax.text(mx + 0.06, ax.get_ylim()[0], f"NPL trung vị {mx:.2f}%", color=INK_3, fontsize=8.5, va="bottom")
ax.text(0.60, my + 0.04, f"NIM trung vị {my:.2f}%", transform=ax.get_yaxis_transform(),
        color=INK_3, fontsize=8.5, va="bottom")
nhan_diem(ax, qq["npl_percent"], qq["nim_ttm_percent"], [TICKER[b] for b in qq["bank_key"]],
          radii=np.sqrt(sizes / np.pi))
ax.set_xlabel("Tỷ lệ nợ xấu NPL (%)")
ax.set_ylabel("NIM lũy kế 4 quý (%)")
moc = [0.25, 0.6, 1.0]
lg_size = ax.legend([ax.scatter([], [], s=f * S_MAX + 40, color=INK_3, alpha=0.35,
                                edgecolor=SURFACE, linewidth=2) for f in moc],
                    [f"{f * qq['du_no'].max():,.0f}" for f in moc],
                    title="Dư nợ (nghìn tỷ đồng)", title_fontsize=9, alignment="left",
                    loc="upper left", bbox_to_anchor=(1.01, 1), labelspacing=1.6,
                    borderpad=1, handletextpad=1.8)
ax.add_artist(lg_size)
chu_thich_nhom(ax, loc="upper left", bbox_to_anchor=(1.01, 0.45))
frame(ax, "Rủi ro và sinh lời đi cùng nhau?",
      f"Quý {Q} · độ lớn bong bóng = dư nợ cho vay · {len(qq)}/22 ngân hàng",
      "Góc dưới phải là vùng xấu nhất: nợ xấu cao mà biên lãi thuần vẫn mỏng. "
      "Đổi biến Q ở ô trên để xem quý khác.")
save(fig, "fig_11_npl_vs_nim")
plt.show()

### 7.2 Quy mô: dư nợ so với tiền gửi

Hai cột cạnh nhau (không xếp chồng) để đọc được cả quy mô lẫn tỷ lệ cho vay trên huy động —
cột cho vay dài hơn cột tiền gửi nghĩa là LDR trên 100%.

In [ ]:
sz = (fact_quarter[fact_quarter["period"] == Q]
      .sort_values("total_assets_ty_vnd")
      [["bank_key", "loans_ty_vnd", "deposits_ty_vnd", "ldr_computed_percent"]])
sz[["cho_vay", "tien_gui"]] = sz[["loans_ty_vnd", "deposits_ty_vnd"]] / NGHIN_TY
y = np.arange(len(sz))
h = 0.38

fig, ax = plt.subplots(figsize=(9.5, 9))
ax.barh(y + h / 2 + 0.01, sz["tien_gui"], height=h, color=SERIES[0], label="Tiền gửi khách hàng")
ax.barh(y - h / 2 - 0.01, sz["cho_vay"], height=h, color=SERIES[1], label="Dư nợ cho vay")
for i, (_, r) in enumerate(sz.iterrows()):
    ax.text(max(r["cho_vay"], r["tien_gui"]) * 1.02, i, f"LDR {r['ldr_computed_percent']:.0f}%",
            va="center", fontsize=8.5, color=INK_3)
ax.set_yticks(y, [NAME[b] for b in sz["bank_key"]])
ax.set_xlabel("Nghìn tỷ đồng")
ax.xaxis.set_major_formatter(lambda v, p: f"{v:,.0f}")
ax.set_xlim(0, max(sz["cho_vay"].max(), sz["tien_gui"].max()) * 1.22)
ax.grid(axis="y", visible=False)
ax.legend(loc="lower right")
frame(ax, "Quy mô cho vay và huy động",
      f"Quý {Q} · sắp theo tổng tài sản · LDR = dư nợ / tiền gửi (tự tính từ BCTC)",
      "Cột *_ty_vnd của fact_bank_quarterly có đơn vị TỶ đồng; ở đây chia thêm 1.000 để hiển thị "
      "bằng nghìn tỷ cho nhãn trục đọc được.")
save(fig, "fig_12_quy_mo_cho_vay_huy_dong")
plt.show()

## 8. Hiệu quả kinh doanh — khai thác `fact_financial`

`fact_financial` là bảng **rộng nhất mà cũng dễ bỏ quên nhất**: 35.556 dòng, 40 chỉ tiêu,
22 ngân hàng, 2018-Q1 → 2026-Q2. Bốn chỉ tiêu dưới đây **phủ 100%** và không có ở bất kỳ bảng
nào khác trong mô hình sao: ROE, ROA, CIR (chi phí trên thu nhập) và P/B.

**Bẫy #2 áp dụng ở đây.** Bảng có cả `freq = quarter` lẫn `freq = year`, kỳ năm có `period_end`
là 31/12 trùng với quý 4 — cộng mà không lọc `freq` là đếm hai lần. `chi_tieu()` luôn lọc sẵn.

**Ba chỉ tiêu KHÔNG dùng được, đã kiểm tra từng cái:**

| Chỉ tiêu | Vấn đề |
|---|---|
| `RT_BANK_CAR` | 654 dòng "có giá trị" nhưng **564 dòng bằng đúng 0** — số 0 giả danh dữ liệu. Vẽ ra sẽ y hệt cái bẫy CAR mà README cảnh báo. |
| `RT_VALUE_EPS`, `RT_VALUE_BVPS` | rỗng hoàn toàn (0/654) |
| `IS_OPERATING_EXPENSES`, `CF_..._FINANCING_ACTIVITIES` | rỗng hoàn toàn (0/748) |

`RT_VALUE_PE` có giá trị nhưng biên độ từ −1.015 đến 937.746 (ngân hàng lỗ hoặc lãi sát 0 làm
mẫu số nổ) nên chỉ dùng P/B.

In [ ]:
def chi_tieu(metric_id: str, freq: str = "quarter", nhan_100: bool = False) -> pd.DataFrame:
    """Lấy một chỉ tiêu từ fact_financial. nhan_100: nguồn ghi dạng thập phân (0,17 = 17%)."""
    d = fact_fin[(fact_fin["metric_id"] == metric_id) & (fact_fin["freq"] == freq)].copy()
    if nhan_100:
        d["value"] = d["value"] * 100
    return d.sort_values("period_end")


QF = fact_fin.loc[fact_fin["freq"] == "quarter", "period"].max()
print("Quý gần nhất của fact_financial:", QF)
print(f"{fact_fin['metric_id'].nunique()} chỉ tiêu · {len(fact_fin):,} dòng · "
      f"{fact_fin['bank_key'].nunique()} ngân hàng")

### 8.1 Sinh lời: ROE và ROA

Hai chỉ tiêu khác đơn vị *nhau về độ lớn* (ROE quanh 17%, ROA quanh 1,5%) nên **không nhét chung
một trục**: vẽ hai panel cạnh nhau, mỗi panel một thang đo, cùng thứ tự ngân hàng để mắt dò ngang.

In [ ]:
roe = chi_tieu("RT_PRT_ROE", nhan_100=True).groupby("bank_key").tail(1).set_index("bank_key")["value"]
roa = chi_tieu("RT_PRT_ROA", nhan_100=True).groupby("bank_key").tail(1).set_index("bank_key")["value"]
thu_tu = roe.sort_values().index

fig, (axL, axR) = plt.subplots(1, 2, figsize=(12, 7), sharey=True,
                               gridspec_kw={"wspace": 0.06})
for ax, s, ten, dv in [(axL, roe, "ROE — sinh lời trên vốn chủ", 1),
                       (axR, roa, "ROA — sinh lời trên tổng tài sản", 1)]:
    v = s.reindex(thu_tu)
    ax.barh([NAME[b] for b in thu_tu], v, height=0.66, color=[mau(b) for b in thu_tu])
    for y, x in enumerate(v):
        ax.text(x + max(v) * 0.015, y, f"{x:.1f}%", va="center", fontsize=8.5, color=INK_2)
    ax.axvline(v.median(), color=INK_3, lw=1)
    ax.set_xlim(min(0, v.min() * 1.1), max(v) * 1.18)
    ax.set_xlabel(f"{ten.split(' — ')[0]} (%)")
    ax.set_title(ten, fontsize=10.5, color=INK, loc="left", pad=8)
    ax.grid(axis="y", visible=False)
chu_thich_nhom(axL, loc="lower right")
# Tiêu đề đặt ở mức figure để không đè lên tiêu đề của hai panel
fig.text(0.005, 1.055, "Ngân hàng nào sinh lời tốt nhất?", fontsize=13, fontweight="600", va="top")
fig.text(0.005, 1.012, f"Quý {QF} · nguồn fact_financial (RT_PRT_ROE / RT_PRT_ROA) · độ đầy 100%",
         fontsize=9.5, color=INK_2, va="top")
ghi_chan(fig, "Hai panel riêng vì ROE và ROA lệch nhau cả chục lần về độ lớn — nhét chung một trục "
              "thì cột ROA sẽ dẹt thành một vạch. Vạch dọc là trung vị của từng chỉ tiêu.")
save(fig, "fig_13_roe_roa")
plt.show()

### 8.2 ROE có bền không? — lưới nhỏ 2018 → nay

Một quý ROE cao chưa nói lên gì. Lưới nhỏ cho thấy ngân hàng nào giữ được mức sinh lời và ngân
hàng nào chỉ cao nhất thời — đúng thứ cần cho bước dự báo và xếp hạng về sau.

In [ ]:
roe_ts = chi_tieu("RT_PRT_ROE", nhan_100=True)
luoi_nho(roe_ts, "period_end", "value",
         "ROE theo quý — từng ngân hàng",
         f"2018-Q1 → {QF} · nguồn fact_financial · ô sắp theo tổng tài sản · số cạnh tên là ROE quý gần nhất",
         "Ô có đường tụt sâu dưới 0 là quý lỗ. Thang đo chung cho cả 22 ô nên so ngang được.",
         "fig_13b_luoi_roe", yfmt=lambda v, p: f"{v:.0f}%")

### 8.3 Sinh lời có đi cùng tiết kiệm chi phí?

**CIR** (Cost to Income Ratio) = chi phí hoạt động / tổng thu nhập hoạt động — càng thấp càng
hiệu quả. Nguồn ghi CIR ở dạng âm (vì chi phí ghi âm trong BCTC) nên phải lấy trị tuyệt đối.

In [ ]:
cir = chi_tieu("RT_BANK_CIR").assign(value=lambda d: d["value"].abs() * 100)
cir_last = cir.groupby("bank_key").tail(1).set_index("bank_key")["value"]
sc2 = pd.DataFrame({"cir": cir_last, "roe": roe}).dropna()

fig, ax = plt.subplots(figsize=(9.5, 6.6))
ax.scatter(sc2["cir"], sc2["roe"], s=95, color=[mau(b) for b in sc2.index],
           edgecolor=SURFACE, linewidth=2, zorder=3)
mx, my = sc2["cir"].median(), sc2["roe"].median()
ax.axvline(mx, color=INK_3, lw=1)
ax.axhline(my, color=INK_3, lw=1)
ax.margins(x=0.10, y=0.14)
ax.text(mx + 0.4, ax.get_ylim()[0], f"CIR trung vị {mx:.1f}%", color=INK_3, fontsize=8.5, va="bottom")
ax.text(0.60, my + 0.3, f"ROE trung vị {my:.1f}%", transform=ax.get_yaxis_transform(),
        color=INK_3, fontsize=8.5, va="bottom")
nhan_diem(ax, sc2["cir"], sc2["roe"], [TICKER[b] for b in sc2.index],
          radii=[np.sqrt(95 / np.pi)] * len(sc2))
ax.set_xlabel("CIR — chi phí hoạt động / tổng thu nhập (%), càng thấp càng tốt")
ax.set_ylabel("ROE (%)")
ax.xaxis.set_major_formatter(lambda v, p: f"{v:.0f}%")
ax.yaxis.set_major_formatter(lambda v, p: f"{v:.0f}%")
chu_thich_nhom(ax, loc="lower left")
frame(ax, "Chi phí gọn có đi cùng sinh lời cao?",
      f"Quý {QF} · nguồn fact_financial (RT_BANK_CIR, RT_PRT_ROE) · độ đầy 100%",
      "Góc trên trái là vùng tốt nhất: chi phí gọn mà vẫn sinh lời cao. "
      "CIR lấy trị tuyệt đối vì BCTC ghi chi phí hoạt động ở dạng âm.")
save(fig, "fig_14_roe_vs_cir")
plt.show()

### 8.4 Ngân hàng sống bằng gì? — cơ cấu thu nhập

Bốn nguồn thu cộng lại bằng tổng thu nhập hoạt động, nên đây là chỗ **đúng** để xếp chồng cột
(khác với COF và lợi suất tài sản ở mục trước — hai chỉ tiêu đó không cộng được với nhau).
Ngân hàng phụ thuộc gần như hoàn toàn vào thu lãi thì lợi nhuận nhạy hơn với biến động lãi suất.

In [ ]:
NGUON_THU = {
    "IS_NET_INTEREST_INCOME": "Thu lãi thuần",
    "IS_NET_FEE_AND_COMMISSION_INCOME": "Thu dịch vụ",
    "IS_NET_GAIN_LOSS_FROM_INVESTMENT_SECURITIES": "Kinh doanh chứng khoán",
    "IS_NET_OTHER_INCOME": "Thu khác",
}
nam_gan = int(QF[:4]) - 1                      # năm tài chính trọn vẹn gần nhất
thu = pd.concat([chi_tieu(m, freq="year").assign(nguon=t) for m, t in NGUON_THU.items()])
thu = thu[thu["period"] == str(nam_gan)]
piv = (thu.pivot_table(index="bank_key", columns="nguon", values="value", aggfunc="sum")
       .reindex(columns=list(NGUON_THU.values())).fillna(0)
       .clip(lower=0))                          # nguồn thu âm (lỗ) dồn về 0 để tỷ trọng cộng đủ 100%
ty_trong = piv.div(piv.sum(axis=1), axis=0) * 100
ty_trong = ty_trong.sort_values("Thu lãi thuần")

fig, ax = plt.subplots(figsize=(10, 7))
trai = np.zeros(len(ty_trong))
khe = 0.35
for i, c in enumerate(ty_trong.columns):
    ax.barh([NAME[b] for b in ty_trong.index], ty_trong[c], left=trai + i * khe,
            height=0.66, color=SERIES[i], label=c)
    trai = trai + ty_trong[c].values
for y, v in enumerate(ty_trong["Thu lãi thuần"]):
    ax.text(2, y, f"{v:.0f}%", va="center", fontsize=8.5, color="#ffffff", fontweight="600")
ax.set_xlabel("Tỷ trọng trong tổng thu nhập hoạt động (%)")
ax.xaxis.set_major_formatter(lambda v, p: f"{v:.0f}%")
ax.set_xlim(0, 103)
ax.grid(axis="y", visible=False)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.075), ncol=4)
frame(ax, f"Ngân hàng sống bằng nguồn thu nào? (năm {nam_gan})",
      "Nguồn fact_financial · freq = year · độ đầy 100% · sắp theo tỷ trọng thu lãi thuần",
      "Bốn nguồn cộng lại bằng 100% nên xếp chồng là đúng. Số in trắng là tỷ trọng thu lãi thuần — "
      "càng cao thì lợi nhuận càng nhạy với biến động lãi suất. Phần âm (lỗ chứng khoán) bị dồn về "
      "0 trong cách tính tỷ trọng này.")
save(fig, "fig_15_co_cau_thu_nhap")
plt.show()

### 8.5 Thị trường định giá ra sao? — P/B

P/B = thị giá / giá trị sổ sách. Không phải chỉ tiêu để chọn nơi gửi tiền, nhưng là thước đo
**thị trường tin ngân hàng nào đến đâu** — bổ sung cho phần giá cổ phiếu ở dưới.

In [ ]:
pb = chi_tieu("RT_VALUE_PB").groupby("bank_key").tail(1).set_index("bank_key")["value"]
pb = pb[pb > 0].sort_values()

fig, ax = plt.subplots(figsize=(9, 6.6))
ax.barh([NAME[b] for b in pb.index], pb.values, height=0.66, color=[mau(b) for b in pb.index])
for y, v in enumerate(pb.values):
    ax.text(v + 0.03, y, f"{v:.2f}", va="center", fontsize=8.5, color=INK_2)
ax.axvline(1, color=INK_3, lw=1)
ax.text(1.03, -1.1, "P/B = 1 · thị giá bằng giá trị sổ sách", color=INK_3, fontsize=8.2, va="center")
ax.set_xlim(0, pb.max() * 1.15)
ax.set_xlabel("P/B (lần)")
ax.grid(axis="y", visible=False)
chu_thich_nhom(ax, loc="lower right")
frame(ax, "Thị trường trả bao nhiêu cho một đồng vốn sổ sách?",
      f"Quý {QF} · nguồn fact_financial (RT_VALUE_PB) · {len(pb)}/22 ngân hàng có giá trị > 0",
      "Không dùng P/E vì biên độ của nó trong dữ liệu này chạy từ −1.015 đến 937.746 — "
      "ngân hàng lỗ hoặc lãi sát 0 làm mẫu số nổ, một biểu đồ cột sẽ vô nghĩa.")
save(fig, "fig_16_pb")
plt.show()

## 9. Trải nghiệm số và mạng lưới

### 9.1 Điểm app ngân hàng số

Điểm tháng phải lấy **bình quân có trọng số theo số lượt đánh giá**: trung bình cộng đơn giản
sẽ cho tháng có 3 đánh giá cùng sức nặng với tháng có 3.000. Chỉ dùng các tháng `usable = True`
(đã loại các tháng bị "review bombing" và tháng quá thưa).

In [ ]:
ap = fact_app[fact_app["usable"]].sort_values("month_date").copy()
ap["w"] = ap["n_reviews"] * ap["mean_score"]


def cuon12(g):
    """Bình quân có trọng số, cửa sổ trượt 12 tháng."""
    g = g.set_index("month_date").sort_index()
    return (g["w"].rolling(12, min_periods=6).sum()
            / g["n_reviews"].rolling(12, min_periods=6).sum()).rename("diem")


roll = (ap.groupby("bank_key", group_keys=True).apply(cuon12, include_groups=False)
        .reset_index().dropna())
roll = roll[roll["month_date"] >= "2019-01-01"]

now = roll.sort_values("month_date").groupby("bank_key").tail(1).set_index("bank_key")["diem"]
dai = roll.groupby("bank_key")["month_date"].count()
uv = now[now.index.isin(dai[dai >= dai.max() * 0.6].index)]        # tránh nhấn một đoạn cụt
HL3 = dict(zip(uv.nlargest(2).index.tolist() + uv.nsmallest(1).index.tolist(), SERIES[:3]))

fig, ax = plt.subplots(figsize=(10.5, 5.6))
for b, g in roll.groupby("bank_key"):
    if b not in HL3:
        ax.plot(g["month_date"], g["diem"], color=MUTED, lw=1, alpha=0.8, zorder=1)
cuoi = []
for b, c in HL3.items():
    g = roll[roll["bank_key"] == b]
    ax.plot(g["month_date"], g["diem"], color=c, lw=2, zorder=3)
    r = g.iloc[-1]
    cuoi.append((r["month_date"], r["diem"], f"{NAME[b]} {r['diem']:.2f}", c))
ax.legend([Line2D([], [], color=c, lw=2) for c in HL3.values()] + [Line2D([], [], color=MUTED, lw=1)],
          [NAME[b] for b in HL3] + [f"{roll['bank_key'].nunique() - len(HL3)} ngân hàng còn lại"],
          loc="lower left", ncol=4)
ax.set_ylim(1, 5.3)
ax.set_ylabel("Điểm trung bình (1–5 sao)")
ax.set_xlim(roll["month_date"].min(), roll["month_date"].max())
frame(ax, "Điểm app ngân hàng số trên Google Play",
      "Bình quân có trọng số theo số lượt, cửa sổ trượt 12 tháng · chỉ tháng usable = True",
      "Cửa sổ trượt để một đợt cập nhật hỏng không kéo nguyên đường xuống đáy. "
      "Đây là dữ liệu PHÍA NGƯỜI DÙNG, không phải chỉ tiêu tài chính.")
cho_nhan(ax, [it[2] for it in cuoi])
nhan_cuoi(ax, cuoi)
save(fig, "fig_17_diem_app_theo_thoi_gian")
plt.show()

In [ ]:
luoi_nho(roll, "month_date", "diem",
         "Điểm app theo thời gian — từng ngân hàng",
         "Bình quân có trọng số, cửa sổ trượt 12 tháng · ô sắp theo tổng tài sản · "
         "số cạnh tên là điểm mới nhất",
         "Đường bắt đầu muộn nghĩa là app đó chưa đủ 6 tháng usable để tính cửa sổ trượt, "
         "không phải điểm bằng 0.",
         "fig_17b_luoi_diem_app")

### 9.2 Phân bố số sao — mở tới từng đánh giá lẻ

Mục trước chỉ dùng bảng tháng đã tổng hợp. `fact_app_review` là **532.007 dòng đánh giá lẻ**,
mỗi dòng có số sao, nội dung, phiên bản app, ngày. Điểm trung bình 3,2 sao có thể là "ai cũng
chấm 3" hoặc "một nửa chấm 5, một nửa chấm 1" — hai tình huống hoàn toàn khác nhau mà điểm
trung bình giấu mất. Phân bố sao mới nói ra điều đó.

Thang màu ở đây là **thang thứ bậc một sắc** (1 sao nhạt → 5 sao đậm) chứ không phải màu phân
loại: số sao có thứ tự tự nhiên.

In [ ]:
NAM_GAN = 2                                     # số năm gần nhất lấy đánh giá
reviews = load("fact_app_review", dates=["date"])
reviews = reviews[reviews["bank_key"].isin(LISTED)]
moc_tu = reviews["date"].max() - pd.DateOffset(years=NAM_GAN)
gan = reviews[reviews["date"] >= moc_tu]

sao = (gan.pivot_table(index="bank_key", columns="score", values="review_id",
                       aggfunc="count").fillna(0))
tong_sao = sao.sum(axis=1)
ty = sao.div(tong_sao, axis=0) * 100
ty = ty.loc[(ty[5] - ty[1]).sort_values().index]        # sắp theo "5 sao trừ 1 sao"

fig, ax = plt.subplots(figsize=(10, 7))
trai = np.zeros(len(ty))
mau_sao = [SEQ(v) for v in (0.16, 0.34, 0.52, 0.72, 0.95)]
for i, c in enumerate(sorted(ty.columns)):
    ax.barh([NAME[b] for b in ty.index], ty[c], left=trai + i * 0.3, height=0.66,
            color=mau_sao[i], label=f"{int(c)} sao")
    trai = trai + ty[c].values
for y, b in enumerate(ty.index):
    ax.text(101.5, y, f"{int(tong_sao[b]):,}", va="center", fontsize=8, color=INK_3)
ax.text(101.5, len(ty) - 0.35, "số đánh giá", va="center", fontsize=8, color=INK_3)
ax.set_xlim(0, 118)
ax.set_xlabel("Tỷ trọng đánh giá (%)")
ax.xaxis.set_major_formatter(lambda v, p: f"{v:.0f}%" if v <= 100 else "")
ax.grid(axis="y", visible=False)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.075), ncol=5)
frame(ax, "Phân bố số sao, không chỉ điểm trung bình",
      f"{len(gan):,} đánh giá trong {NAM_GAN} năm gần nhất · nguồn fact_app_review · "
      f"sắp theo (tỷ lệ 5 sao − tỷ lệ 1 sao)",
      "Điểm trung bình giấu mất hình dạng phân bố: 3,2 sao có thể là ai cũng chấm 3, cũng có thể là "
      "một nửa chấm 5 và một nửa chấm 1. Cột 1 sao dài là dấu hiệu có sự cố thật, không phải chê vặt.")
save(fig, "fig_18_phan_bo_sao_app")
plt.show()

### 9.3 Quy mô người dùng app

`dim_bank` có sẵn ba cột chưa dùng: `app_score`, `app_ratings`, `app_installs` — ảnh chụp cấp app
tại lần cào gần nhất. Lượt cài đặt là **bậc thang do Google Play công bố** (100k, 500k, 1tr,
5tr, 10tr…) chứ không phải con số chính xác, nên đọc như bậc chứ đừng đọc như số đo.

In [ ]:
app = dim_bank[dim_bank["is_listed"]].dropna(subset=["app_installs", "app_score"]).copy()
app = app.sort_values("app_installs")

fig, ax = plt.subplots(figsize=(9.5, 6.4))
ax.scatter(app["app_installs"], app["app_score"], s=95,
           color=[mau(b) for b in app["bank_key"]], edgecolor=SURFACE, linewidth=2, zorder=3)
ax.set_xscale("log")
ax.margins(x=0.14, y=0.16)
nhan_diem(ax, app["app_installs"], app["app_score"], list(app["ticker"]),
          radii=[np.sqrt(95 / np.pi)] * len(app))
ax.set_xlabel("Lượt cài đặt trên Google Play (bậc thang, trục log)")
ax.set_ylabel("Điểm hiển thị trên store (1–5 sao)")
ax.xaxis.set_major_formatter(lambda v, p: f"{v/1e6:g} tr" if v >= 1e6 else f"{v/1e3:g}k")
chu_thich_nhom(ax, loc="lower right")
frame(ax, "App nhiều người dùng có được chấm cao hơn?",
      f"{len(app)}/22 ngân hàng có số liệu app · nguồn dim_bank (ảnh chụp Google Play)",
      "Trục hoành là log vì lượt cài chênh nhau tới trăm lần. Điểm hiển thị trên store là điểm "
      "tích lũy từ lúc ra app, khác với điểm 12 tháng gần nhất ở mục trước — hai con số không "
      "phải một.")
save(fig, "fig_19_quy_mo_app")
plt.show()

### 9.4 Mạng lưới điểm giao dịch

**Bản đồ dùng lat/lon, không dùng tên tỉnh.** `dim_province` là 34 tỉnh sau sáp nhập 2025, kèm
tiền tố "Thành phố"/"Tỉnh" và đủ dấu tiếng Việt — geocode theo tên hay trả sai. `fact_branch_point`
có lat/lon đủ 5.904/5.904 dòng nên không cần geocode gì cả.

Lọc `is_listed = True` bỏ **2.354 điểm `UNKNOWN`** mà OSM không gắn thương hiệu; để nguyên thì
chúng át hết 3.550 điểm của 22 ngân hàng.

In [ ]:
pt = fact_point.dropna(subset=["lat", "lon"])
fig, ax = plt.subplots(figsize=(5.6, 8.6))
ax.scatter(pt["lon"], pt["lat"], s=7, color=SERIES[0], alpha=0.30, linewidth=0)
ax.set_aspect(1 / np.cos(np.radians(pt["lat"].mean())))
ax.set_xlabel("Kinh độ")
ax.set_ylabel("Vĩ độ")
frame(ax, "Mạng lưới điểm giao dịch và ATM",
      f"{len(pt):,} điểm của 22 ngân hàng niêm yết · nguồn OpenStreetMap",
      "OSM ghi nhận thiếu ở vùng ít người đóng góp, nên bản đồ này phản ánh cả độ phủ của dữ liệu "
      "lẫn độ phủ thật của ngân hàng.")
save(fig, "fig_20_ban_do_diem_giao_dich")
plt.show()

### 9.5 Ngân hàng nào nhiều điểm nhất?

In [ ]:
net = (fact_branch.groupby("bank_key")[["n_bank", "n_atm"]].sum()
       .assign(tong=lambda d: d["n_bank"] + d["n_atm"]).sort_values("tong"))
khe = net["tong"].max() * 0.004                       # khe hở 2px giữa hai phần

fig, ax = plt.subplots(figsize=(9, 7))
ten = [NAME[b] for b in net.index]
ax.barh(ten, net["n_bank"], height=0.66, color=SERIES[0], label="Chi nhánh / PGD")
ax.barh(ten, net["n_atm"], height=0.66, left=net["n_bank"] + khe, color=SERIES[1], label="ATM")
for i, v in enumerate(net["tong"]):
    ax.text(v + net["tong"].max() * 0.015, i, f"{int(v)}", va="center", fontsize=8.5, color=INK_2)
ax.set_xlabel("Số điểm ghi nhận trên OpenStreetMap")
ax.set_xlim(0, net["tong"].max() * 1.12)
ax.grid(axis="y", visible=False)
ax.legend(loc="lower right")
frame(ax, "Số điểm giao dịch và ATM theo ngân hàng",
      f"{int(net['tong'].sum()):,} điểm · 34 tỉnh/thành sau sáp nhập 2025 · nguồn OpenStreetMap",
      "Hai phần cách nhau một khe hở nhỏ thay vì viền — mắt vẫn tách được mà không thêm nét.")
save(fig, "fig_21_mang_luoi_theo_ngan_hang")
plt.show()

### 9.6 Mạng lưới theo tỉnh — chiều `province` chưa dùng

`fact_branch_by_province` được dùng ở mục trước nhưng đã cộng dồn mất chiều tỉnh. Ma trận
tỉnh × ngân hàng trả lời câu hỏi rất thực tế của người chọn ngân hàng: **ở tỉnh của tôi thì
ngân hàng nào có điểm giao dịch?**

In [ ]:
TOP_TINH = 12
tinh_lon = (fact_branch.groupby("province")["n_total"].sum()
            .sort_values(ascending=False).head(TOP_TINH).index)
mt = (fact_branch[fact_branch["province"].isin(tinh_lon)]
      .pivot_table(index="province", columns="bank_key", values="n_total", aggfunc="sum")
      .reindex(index=tinh_lon)
      .reindex(columns=sorted(LISTED, key=lambda b: -_ts.get(b, 0))))

fig, ax = plt.subplots(figsize=(11, 5.4))
vals = mt.values.astype(float)
im = ax.imshow(np.where(np.isnan(vals), 0, vals), cmap=SEQ, aspect="auto",
               vmin=0, vmax=np.nanpercentile(vals, 96))
ax.set_xticks(range(mt.shape[1]), [TICKER[b] for b in mt.columns], fontsize=8.5)
ax.set_yticks(range(mt.shape[0]), [t.replace("Thành phố ", "TP ").replace("Tỉnh ", "")
                                   for t in mt.index], fontsize=9)
ax.set_xticks(np.arange(-0.5, mt.shape[1], 1), minor=True)
ax.set_yticks(np.arange(-0.5, mt.shape[0], 1), minor=True)
ax.grid(which="minor", color=SURFACE, lw=2)
ax.grid(which="major", visible=False)
ax.tick_params(which="minor", length=0)
nguong = np.nanpercentile(vals, 96) * 0.55
for i in range(mt.shape[0]):
    for j in range(mt.shape[1]):
        v = vals[i, j]
        ax.text(j, i, "·" if np.isnan(v) else f"{int(v)}", ha="center", va="center", fontsize=7.5,
                color=INK_3 if np.isnan(v) else ("#ffffff" if v > nguong else INK))
cb = fig.colorbar(im, ax=ax, pad=0.015, fraction=0.025)
cb.set_label("số điểm", color=INK_2, fontsize=9)
cb.outline.set_visible(False)
frame(ax, f"{TOP_TINH} tỉnh/thành nhiều điểm giao dịch nhất",
      "Chi nhánh/PGD + ATM · nguồn OpenStreetMap · cột sắp theo tổng tài sản ngân hàng",
      "Ô '·' là OSM không ghi nhận điểm nào của ngân hàng đó ở tỉnh đó — có thể là chưa ai đóng góp "
      "dữ liệu chứ không chắc là ngân hàng không có mặt. Thang màu cắt ở phân vị 96 để Hà Nội và "
      "TP.HCM không nuốt hết dải màu.")
save(fig, "fig_22_mang_luoi_theo_tinh")
plt.show()
# ## 10. Giá cổ phiếu — chuẩn hóa về cùng gốc 100
#
# Giá tuyệt đối của 22 mã chênh nhau hàng chục lần nên vẽ chung sẽ không đọc được. Chuẩn hóa
# **cùng một gốc = 100** đưa tất cả về một trục duy nhất — đây cũng là cách thay thế đúng cho
# biểu đồ hai trục.

In [ ]:
BASE = "2023-01-01"
px = fact_price[fact_price["date"] >= BASE].sort_values("date")
px = px.assign(idx=px["close"] / px.groupby("bank_key")["close"].transform("first") * 100)

het = px.sort_values("date").groupby("bank_key").tail(1).set_index("bank_key")["idx"]
HL4 = dict(zip(het.nlargest(2).index.tolist() + het.nsmallest(1).index.tolist(), SERIES[:3]))

fig, ax = plt.subplots(figsize=(10.5, 5.6))
for b, g in px.groupby("bank_key"):
    if b not in HL4:
        ax.plot(g["date"], g["idx"], color=MUTED, lw=0.9, alpha=0.75, zorder=1)
cuoi = []
for b, c in HL4.items():
    g = px[px["bank_key"] == b]
    ax.plot(g["date"], g["idx"], color=c, lw=2, zorder=3)
    r = g.iloc[-1]
    cuoi.append((r["date"], r["idx"], f"{TICKER[b]} {r['idx']:.0f}", c))
ax.axhline(100, color=INK_3, lw=1)
ax.legend([Line2D([], [], color=c, lw=2) for c in HL4.values()] + [Line2D([], [], color=MUTED, lw=1)],
          [TICKER[b] for b in HL4] + [f"{px['bank_key'].nunique() - len(HL4)} mã còn lại"],
          loc="upper left", ncol=4)
ax.set_ylabel(f"Chỉ số giá (phiên đầu tiên từ {BASE} = 100)")
ax.set_xlim(px["date"].min(), px["date"].max())
frame(ax, "Giá cổ phiếu ngân hàng, chuẩn hóa về gốc 100",
      f"{BASE} → {px['date'].max():%d/%m/%Y} · giá đóng cửa đã điều chỉnh (vnstock)",
      "Chuẩn hóa về cùng gốc cho phép 22 mã dùng chung MỘT trục y — không bao giờ dùng hai trục y "
      "để nhét hai thang đo vào một khung.")
cho_nhan(ax, [it[2] for it in cuoi])
nhan_cuoi(ax, cuoi)
save(fig, "fig_23_gia_co_phieu_chuan_hoa")
plt.show()

Hình trên chỉ đọc được 3 mã — nó trả lời "cả ngành đi về đâu, biên độ rộng cỡ nào". Muốn biết
**một mã cụ thể** đi ra sao thì phải sang lưới nhỏ. Dữ liệu giá là hàng ngày (47.206 dòng) nên
lấy giá cuối tuần cho nhẹ; hình dạng đường không đổi ở độ phân giải này.

In [ ]:
tuan = (px.set_index("date").groupby("bank_key")["idx"].resample("W").last()
        .reset_index().dropna())
luoi_nho(tuan, "date", "idx",
         "Giá cổ phiếu chuẩn hóa về gốc 100 — từng mã",
         f"{BASE} → {px['date'].max():%d/%m/%Y} · giá cuối tuần · ô sắp theo tổng tài sản · "
         f"số cạnh tên là chỉ số giá mới nhất",
         "Vạch 100 là mốc gốc: đường nằm dưới nghĩa là mã đó vẫn thấp hơn giá đầu kỳ. "
         "Nền xám là 21 mã còn lại, thang đo chung cho cả 22 ô.",
         "fig_23b_luoi_gia_co_phieu", moc_y=100)

## 11. Chuẩn tham chiếu — biến dữ liệu thành thước đo

Các mục trước trả lời từng câu hỏi rời rạc. Mục này gom **toàn bộ** dữ liệu dùng được thành một
bộ tiêu chí chung rồi dựng **mốc quy chiếu** cho từng tiêu chí: thế nào là cao, thế nào là thấp,
ngân hàng đang đứng ở đâu so với mặt bằng 22 mã.

Đây là bước nối sang phần xếp hạng TOPSIS/AHP: TOPSIS cần đúng ba thứ mà mục này dựng ra —
**ma trận quyết định** (ngân hàng × tiêu chí), **chiều** của từng tiêu chí (cao hơn tốt hơn hay
ngược lại), và **thang chuẩn hóa** để cộng được các đơn vị khác nhau lại với nhau.

Ba điều phải nói rõ trước:

1. **"Tốt" tùy theo đứng ở phía nào.** Lãi suất tiền gửi cao là tốt cho người gửi nhưng là chi
   phí của ngân hàng; lãi suất cho vay cao thì ngược lại. Mỗi tiêu chí dưới đây đều ghi rõ
   **tốt cho ai**, không gộp bừa vào một chữ "tốt".
2. **Có tiêu chí không có chiều.** LDR quá thấp là ứ vốn, quá cao là căng thanh khoản — nó cần
   một *khoảng* chứ không phải một chiều. Những tiêu chí như vậy đánh dấu chiều `0` và **bị loại
   khỏi điểm tổng hợp**, chỉ giữ lại để tra cứu.
3. **Mốc quy chiếu ở đây là tứ phân vị của chính 22 mã niêm yết**, không phải chuẩn ngành hay
   chuẩn Basel. Nói cách khác: "tốt" nghĩa là tốt *so với 21 ngân hàng còn lại", không phải tốt
   theo một ngưỡng tuyệt đối nào.

### 11.1 Kiểm kê toàn bộ dữ liệu

Trước khi chọn tiêu chí, đếm xem có bao nhiêu cột thực sự dùng được. Mỗi cột số của mỗi bảng
fact được xếp vào ba bậc theo độ đầy: **đủ dùng** (≥ 90%), **một phần** (50–90%), **quá thưa**
(< 50%, chỉ nên tra cứu chứ không nên vẽ hay đưa vào mô hình).

Hai chỗ dễ đếm sai, đã xử lý: `fact_financial` ở **dạng long** (40 chỉ tiêu nằm trên các dòng,
không phải 40 cột) nên đếm theo `metric_id`; và `dim_bank` chỉ lấy 22 mã niêm yết, vì ba cột app
vốn trống ở phần lớn 31 ngân hàng ngoài sàn.

In [ ]:
BANG = {
    "fact_deposit_rate": fact_deposit, "fact_lending_rate_monthly": fact_lending,
    "fact_bank_quarterly": fact_quarter, "fact_financial": fact_fin,
    "fact_app_review_monthly": fact_app, "fact_branch_by_province": fact_branch,
    "fact_branch_point": fact_point, "fact_price_daily": fact_price,
    "fact_macro_monthly": macro, "fact_credit_growth_yearly": credit_growth,
    "dim_bank": dim_bank[dim_bank["is_listed"]],          # chỉ 22 mã, không tính NH ngoài sàn
}
# Bảng dạng long: một chỉ tiêu nằm ở NHIỀU DÒNG chứ không phải một cột. Đếm cột ở đây sẽ ra
# "fact_financial có 1 cột số" — sai đơn vị đo. Với các bảng này, mỗi giá trị của cột khóa là
# một chỉ tiêu riêng.
DANG_LONG = {"fact_financial": ("metric_id", "value", {"freq": "quarter"})}


def bac_do_day(s: pd.Series) -> tuple[float, float, str]:
    phu = s.notna().mean() * 100
    khac0 = (s.fillna(0) != 0).mean() * 100      # cột đầy nhưng toàn số 0 thì vẫn là rỗng
    return phu, khac0, ("đủ dùng" if min(phu, khac0) >= 90 else
                        "một phần" if min(phu, khac0) >= 50 else "quá thưa")


kk = []
for ten, d in BANG.items():
    if ten in DANG_LONG:
        khoa, gia_tri, loc = DANG_LONG[ten]
        for c, v in loc.items():
            d = d[d[c] == v]
        for m, g in d.groupby(khoa):
            phu, khac0, bac = bac_do_day(g[gia_tri])
            kk.append({"bảng": ten, "chỉ tiêu": m, "số dòng": len(g),
                       "độ đầy %": round(phu, 1), "khác 0 %": round(khac0, 1), "bậc": bac})
        continue
    for c in d.columns:
        col = d[c]
        if not pd.api.types.is_numeric_dtype(col) or pd.api.types.is_bool_dtype(col):
            continue
        phu, khac0, bac = bac_do_day(col)
        kk.append({"bảng": ten, "chỉ tiêu": c, "số dòng": len(d),
                   "độ đầy %": round(phu, 1), "khác 0 %": round(khac0, 1), "bậc": bac})
kiem_ke = pd.DataFrame(kk)

BAC = ["đủ dùng", "một phần", "quá thưa"]
MAU_BAC = [SEQ(0.30), SEQ(0.60), SEQ(0.92)]
dem = (kiem_ke.pivot_table(index="bảng", columns="bậc", values="chỉ tiêu", aggfunc="count")
       .reindex(columns=BAC).fillna(0))
dem = dem.loc[dem.sum(axis=1).sort_values().index]

fig, ax = plt.subplots(figsize=(10, 5.6))
trai = np.zeros(len(dem))
for i, b in enumerate(BAC):
    ax.barh(dem.index, dem[b], left=trai + i * 0.06, height=0.66, color=MAU_BAC[i], label=b)
    trai = trai + dem[b].values
for y, (ten, r) in enumerate(dem.iterrows()):
    dv = "chỉ tiêu" if ten in DANG_LONG else "cột số"
    ax.text(r.sum() + 0.5, y, f"{int(r.sum())} {dv} · {len(BANG[ten]):,} dòng",
            va="center", fontsize=8.5, color=INK_3)
ax.set_xlim(0, dem.sum(axis=1).max() * 1.55)
ax.set_xlabel("Số chỉ tiêu đo được trong bảng")
ax.grid(axis="y", visible=False)
ax.legend(loc="lower right", ncol=3)
frame(ax, "Kiểm kê: bảng nào còn chỉ tiêu dùng được?",
      f"{len(kiem_ke)} chỉ tiêu trên {len(BANG)} bảng · bậc xét theo cả độ đầy lẫn tỷ lệ khác 0",
      "fact_financial ở dạng long nên đếm theo metric_id (40 chỉ tiêu) chứ không đếm cột — đếm cột "
      "sẽ ra 'một cột số' cho bảng giàu nhất. Xét thêm 'khác 0 %' vì có chỉ tiêu đầy dữ liệu nhưng "
      "toàn số 0: CAR có 654 dòng giá trị mà 564 dòng bằng đúng 0, vẫn bị xếp vào bậc quá thưa.")
save(fig, "fig_24_kiem_ke_du_lieu")
plt.show()

# 12 chỉ tiêu thưa nhất — những thứ phải loại khỏi mô hình
kiem_ke.sort_values(["bậc", "độ đầy %"], ascending=[False, True]).head(12).reset_index(drop=True)

### 11.2 Ma trận quyết định — 22 ngân hàng × các tiêu chí

`chieu = +1` cao hơn tốt hơn · `-1` thấp hơn tốt hơn · `0` cần một khoảng chứ không có chiều,
không đưa vào điểm tổng hợp.

In [ ]:
qL = fact_quarter[fact_quarter["period"] == Q].set_index("bank_key")
lend_i = lend_last.set_index("bank_key")

TIEU_CHI = [
    # (tên, chuỗi giá trị, chiều, đơn vị, tốt cho ai)
    (f"LS tiền gửi {DEP_TERM}T", rank.set_index("bank_key")["rate"], +1, "%/năm", "Người gửi"),
    ("LS cho vay", lend_i["lending_avg_percent"], -1, "%/năm", "Người vay"),
    ("Chênh lệch vay−gửi", lend_i["spread_vs_deposit_point"], -1, "điểm %", "Người dùng"),
    ("Điểm app", now, +1, "sao", "Người dùng"),
    ("Số đánh giá app", tong_sao, +1, "lượt", "Người dùng"),
    ("Điểm giao dịch", net["tong"], +1, "điểm", "Người dùng"),
    ("ROE", roe, +1, "%", "Sức khỏe NH"),
    ("ROA", roa, +1, "%", "Sức khỏe NH"),
    ("NIM (TTM)", qL["nim_ttm_percent"], +1, "%", "Sức khỏe NH"),
    ("CIR", cir_last, -1, "%", "Sức khỏe NH"),
    ("Nợ xấu NPL", qL["npl_percent"], -1, "%", "An toàn"),
    ("Bao phủ nợ xấu", qL["npl_coverage_percent"], +1, "%", "An toàn"),
    ("Tổng tài sản", qL["total_assets_ty_vnd"] / NGHIN_TY, +1, "nghìn tỷ", "An toàn"),
    ("LDR", qL["ldr_computed_percent"], 0, "%", "cần một khoảng"),
    ("P/B", pb, 0, "lần", "Thị trường"),
    ("Chỉ số giá (gốc 100)", het, 0, "điểm", "Thị trường"),
]

MT = pd.DataFrame({ten: s.reindex(LISTED) for ten, s, *_ in TIEU_CHI}, index=LISTED)
CHIEU = {ten: c for ten, _, c, *_ in TIEU_CHI}
DON_VI = {ten: u for ten, _, _, u, _ in TIEU_CHI}
CHO_AI = {ten: a for ten, *_, a in TIEU_CHI}
XEP_HANG = [t for t in MT.columns if CHIEU[t] != 0]          # tiêu chí có chiều rõ ràng

print(f"Ma trận quyết định: {MT.shape[0]} ngân hàng × {MT.shape[1]} tiêu chí "
      f"({len(XEP_HANG)} tiêu chí có chiều, {MT.shape[1] - len(XEP_HANG)} tiêu chí chỉ tra cứu)")
MT.rename(index=TICKER).round(2)

### 11.3 Bảng mốc quy chiếu

Với mỗi tiêu chí: nhỏ nhất, tứ phân vị, lớn nhất, và **ngưỡng tốt** — mốc mà vượt qua là nằm
trong nhóm 25% dẫn đầu theo chiều đã khai báo (p75 nếu cao hơn tốt hơn, p25 nếu ngược lại).

In [ ]:
moc = []
for t in MT.columns:
    s = MT[t].dropna()
    q1, q2, q3 = s.quantile([0.25, 0.5, 0.75])
    moc.append({
        "Tiêu chí": t, "Đơn vị": DON_VI[t], "Tốt cho": CHO_AI[t],
        "Chiều": {1: "cao hơn ↑", -1: "thấp hơn ↓", 0: "cần khoảng"}[CHIEU[t]],
        "Đủ": f"{len(s)}/22",
        "Nhỏ nhất": s.min(), "p25": q1, "Trung vị": q2, "p75": q3, "Lớn nhất": s.max(),
        "Ngưỡng tốt": q3 if CHIEU[t] == 1 else (q1 if CHIEU[t] == -1 else np.nan),
        "Dẫn đầu": ", ".join(TICKER[b] for b in (
            s.nlargest(2).index if CHIEU[t] == 1 else
            s.nsmallest(2).index if CHIEU[t] == -1 else s.nlargest(2).index)),
    })
QUY_CHIEU = pd.DataFrame(moc).set_index("Tiêu chí")
QUY_CHIEU.round(2)

### 11.4 Từng tiêu chí, 22 ngân hàng nằm ở đâu

Mỗi ô là một tiêu chí: 22 chấm là 22 ngân hàng, vùng xám là khoảng tứ phân vị (50% ở giữa),
vạch đậm là trung vị. **Vạch xanh là ngưỡng tốt** (p75 nếu cao hơn tốt hơn, p25 nếu ngược lại),
vùng xanh nhạt phía sau nó là nhóm 25% dẫn đầu.
Nhìn hình này là thấy ngay tiêu chí nào phân hóa mạnh (chấm trải rộng) và tiêu chí nào cả
ngành gần như nhau (chấm dồn cục — đưa vào mô hình cũng không phân biệt được ai với ai).

In [ ]:
ncols, cot = 2, list(MT.columns)
nrows = -(-len(cot) // ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(6.2 * ncols, 1.15 * nrows))
axes = np.atleast_1d(axes).ravel()

for ax, t in zip(axes, cot):
    s = MT[t].dropna()
    q1, q2, q3 = s.quantile([0.25, 0.5, 0.75])
    bien = (s.max() - s.min()) * 0.08
    ax.axvspan(q1, q3, color=INK_3, alpha=0.08, zorder=0)
    if CHIEU[t] != 0:                                   # vùng vượt ngưỡng + vạch ngưỡng
        nguong = q3 if CHIEU[t] == 1 else q1
        canh = s.max() + bien if CHIEU[t] == 1 else s.min() - bien
        ax.axvspan(*sorted([nguong, canh]), color=STATUS["good"], alpha=0.13, zorder=0)
        ax.axvline(nguong, color=STATUS["good"], lw=1.6, zorder=2)
        ax.annotate(f"ngưỡng tốt {nguong:,.2f}".rstrip("0").rstrip(","),
                    (nguong, 0.5), xycoords=ax.get_xaxis_transform(), xytext=(4, -2),
                    textcoords="offset points", ha="left", va="top", fontsize=7.5,
                    color=STATUS["good"], fontweight="600")
    ax.axvline(q2, color=INK_2, lw=1.2, zorder=2)
    ax.scatter(s, np.zeros(len(s)), s=58, color=[mau(b) for b in s.index],
               alpha=0.85, edgecolor=SURFACE, linewidth=1.5, zorder=3, clip_on=False)
    for b in (s.idxmax(), s.idxmin()):                  # cả hai nhãn đặt phía trên, tránh trục
        ax.annotate(TICKER[b], (s[b], 0), xytext=(0, 12), textcoords="offset points",
                    ha="center", fontsize=7.5, color=INK_3)
    mui = {1: " ↑", -1: " ↓", 0: " ↔"}[CHIEU[t]]
    ax.set_title(f"{t}{mui}   ({DON_VI[t]}) · {CHO_AI[t]}", fontsize=9.5, loc="left",
                 color=INK, pad=6)
    ax.set_yticks([])
    ax.set_ylim(-0.55, 0.55)
    ax.margins(x=0.09)
    ax.grid(axis="y", visible=False)
    ax.spines["left"].set_visible(False)
    ax.tick_params(labelsize=8)
for ax in axes[len(cot):]:
    ax.set_visible(False)

cao = fig.get_figheight()
fig.tight_layout(rect=[0, 0, 1, 1 - 0.72 / cao], h_pad=2.2)
fig.text(0.005, 1 - 0.06 / cao, "Mặt bằng 22 ngân hàng theo từng tiêu chí",
         fontsize=13, fontweight="600", va="top")
fig.text(0.005, 1 - 0.34 / cao,
         "Vùng xám = khoảng tứ phân vị · vạch đậm = trung vị · vùng xanh nhạt = vượt ngưỡng tốt · "
         "màu chấm theo nhóm ngân hàng", fontsize=9.5, color=INK_2, va="top")
ghi_chan(fig, "Mũi tên cạnh tên tiêu chí là chiều: ↑ cao hơn tốt hơn, ↓ thấp hơn tốt hơn, "
              "↔ cần một khoảng nên không có chiều. Chỉ ghi tên mã ở hai đầu để hình khỏi rối; "
              "giá trị đầy đủ nằm ở ma trận quyết định phía trên.", y=-0.004)
save(fig, "fig_25_phan_bo_tieu_chi")
plt.show()

### 11.5 Bản đồ lệch chuẩn — ai trên, ai dưới mặt bằng

Mỗi ô là **z-score đã nhân chiều**: `(giá trị − trung bình) / độ lệch chuẩn × chiều`. Nhờ nhân
chiều nên **dương luôn là tốt** ở mọi tiêu chí, kể cả tiêu chí "thấp hơn tốt hơn" như nợ xấu.
Đây chính là bước chuẩn hóa mà TOPSIS cần: đưa %/năm, nghìn tỷ, số sao về cùng một thang.

**Xanh = trên mặt bằng ngành, đỏ = dưới, xám ở giữa = đúng mức trung bình.** Điểm giữa của thang
không phải màu ngẫu nhiên mà chính là mốc quy chiếu.

In [ ]:
Z = ((MT[XEP_HANG] - MT[XEP_HANG].mean()) / MT[XEP_HANG].std())
Z = Z * pd.Series({t: CHIEU[t] for t in XEP_HANG})           # dương = tốt, ở mọi tiêu chí
diem_tho = Z.mean(axis=1).sort_values()
Zs = Z.loc[diem_tho.index]

fig, ax = plt.subplots(figsize=(11, 7))
gh = np.nanpercentile(np.abs(Zs.values), 98)
im = ax.imshow(Zs.values, cmap=DIV, aspect="auto", vmin=-gh, vmax=gh)
ax.set_xticks(range(Zs.shape[1]), Zs.columns, rotation=35, ha="right", fontsize=8.5)
ax.set_yticks(range(Zs.shape[0]), [NAME[b] for b in Zs.index], fontsize=9)
ax.set_xticks(np.arange(-0.5, Zs.shape[1], 1), minor=True)
ax.set_yticks(np.arange(-0.5, Zs.shape[0], 1), minor=True)
ax.grid(which="minor", color=SURFACE, lw=2)
ax.grid(which="major", visible=False)
ax.tick_params(which="minor", length=0)
for i in range(Zs.shape[0]):
    for j in range(Zs.shape[1]):
        v = Zs.values[i, j]
        ax.text(j, i, "·" if np.isnan(v) else f"{v:+.1f}", ha="center", va="center",
                fontsize=7.5, color=INK_3 if np.isnan(v) else
                ("#ffffff" if abs(v) > gh * 0.62 else INK))
cb = fig.colorbar(im, ax=ax, pad=0.015, fraction=0.022)
cb.set_label("độ lệch chuẩn so với mặt bằng ngành", color=INK_2, fontsize=9)
cb.outline.set_visible(False)
frame(ax, "Ngân hàng nào trên, ai dưới mặt bằng — từng tiêu chí",
      f"z-score đã nhân chiều: dương = tốt · {len(XEP_HANG)} tiêu chí có chiều · "
      f"hàng sắp theo trung bình z",
      "Ô '·' là ngân hàng thiếu số liệu ở tiêu chí đó, KHÔNG phải bằng trung bình ngành — đừng đọc "
      "ô trắng thành 'trung bình'. Thang cắt ở phân vị 98 của |z| để một ngoại lai không nuốt hết dải màu.")
save(fig, "fig_26_z_score")
plt.show()

### 11.6 Tiêu chí nào đang đo trùng nhau?

Bước gán trọng số (AHP) hay bị lỗi này: đưa vào ba tiêu chí gần như đo cùng một thứ thì thứ đó
vô tình được nhân ba lần trọng số. Ma trận tương quan chỉ ra các cặp cần gộp hoặc bỏ bớt.

In [ ]:
tq = MT[XEP_HANG].corr(method="spearman")
fig, ax = plt.subplots(figsize=(8.6, 7.4))
im = ax.imshow(tq.values, cmap=DIV, vmin=-1, vmax=1, aspect="auto")
ax.set_xticks(range(len(tq)), tq.columns, rotation=35, ha="right", fontsize=8.5)
ax.set_yticks(range(len(tq)), tq.index, fontsize=8.5)
ax.set_xticks(np.arange(-0.5, len(tq), 1), minor=True)
ax.set_yticks(np.arange(-0.5, len(tq), 1), minor=True)
ax.grid(which="minor", color=SURFACE, lw=2)
ax.grid(which="major", visible=False)
ax.tick_params(which="minor", length=0)
for i in range(len(tq)):
    for j in range(len(tq)):
        v = tq.values[i, j]
        ax.text(j, i, f"{v:+.1f}".replace("+1.0", "1"), ha="center", va="center", fontsize=7,
                color="#ffffff" if abs(v) > 0.62 else INK)
cb = fig.colorbar(im, ax=ax, pad=0.015, fraction=0.03)
cb.set_label("hệ số tương quan hạng (Spearman)", color=INK_2, fontsize=9)
cb.outline.set_visible(False)
frame(ax, "Tiêu chí nào đang đo trùng nhau?",
      "Tương quan hạng Spearman giữa các tiêu chí có chiều · dùng hạng nên không sợ ngoại lai",
      "Cặp nào |hệ số| > 0,8 thì gần như đo cùng một thứ — giữ cả hai là vô tình nhân đôi trọng số "
      "cho khía cạnh đó ở bước AHP/TOPSIS.")
save(fig, "fig_27_tuong_quan_tieu_chi")
plt.show()

cap = (tq.where(np.triu(np.ones(tq.shape), 1).astype(bool)).stack()
       .rename("tương quan").reset_index())
cap.columns = ["Tiêu chí A", "Tiêu chí B", "Tương quan"]
print("Các cặp trùng nhau nhiều nhất (|hệ số| ≥ 0,7):")
cap.reindex(cap["Tương quan"].abs().sort_values(ascending=False).index).query("abs(`Tương quan`) >= 0.7").round(2)

### 11.7 Điểm tổng hợp thô — mốc để đối chiếu về sau

Trung bình cộng z-score của các tiêu chí có chiều, **trọng số bằng nhau**. Đây **chưa phải**
kết quả TOPSIS/AHP: nó giả định mọi tiêu chí quan trọng như nhau, mà giả định đó gần như chắc
chắn sai — người gửi tiết kiệm không coi "số điểm giao dịch" ngang với "lãi suất".

Giá trị của nó là làm **mốc đối chiếu**: khi mô hình có trọng số cho ra thứ hạng lệch hẳn khỏi
bảng này, phải giải thích được vì sao — hoặc là trọng số đang nói lên điều gì đó thật, hoặc là
mô hình có lỗi.

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 7))
ax.barh([NAME[b] for b in diem_tho.index], diem_tho.values, height=0.66,
        color=[mau(b) for b in diem_tho.index])
ax.axvline(0, color=INK_2, lw=1)
for y, v in enumerate(diem_tho.values):
    ax.text(v + (0.02 if v >= 0 else -0.02), y, f"{v:+.2f}", va="center", fontsize=8.5,
            color=INK_2, ha="left" if v >= 0 else "right")
lim = np.abs(diem_tho).max()
ax.set_xlim(-lim * 1.35, lim * 1.35)
ax.set_xlabel("Trung bình z-score đã nhân chiều (0 = đúng mặt bằng ngành)")
ax.grid(axis="y", visible=False)
chu_thich_nhom(ax, loc="lower right")
frame(ax, "Điểm tổng hợp thô — trọng số bằng nhau",
      f"{len(XEP_HANG)} tiêu chí có chiều · 0 = trung bình ngành · ĐÂY CHƯA PHẢI KẾT QUẢ TOPSIS",
      "Trọng số bằng nhau là giả định tạm, gần như chắc chắn sai — dùng làm mốc đối chiếu cho mô "
      "hình có trọng số ở giai đoạn sau, không dùng làm kết luận của khóa luận.")
save(fig, "fig_28_diem_tong_hop_tho")
plt.show()

### 11.8 Hồ sơ một ngân hàng

Đổi `HO_SO` để xem ngân hàng khác. Thanh phải là mạnh hơn mặt bằng, thanh trái là yếu hơn —
đọc được ngay ngân hàng đó thắng ở đâu và thua ở đâu.

In [ ]:
HO_SO = str(diem_tho.index[-1])          # mặc định: ngân hàng có điểm tổng hợp cao nhất


def ve_ho_so(bank_key: str):
    z = Z.loc[bank_key].dropna().sort_values()
    fig, ax = plt.subplots(figsize=(9, 0.42 * len(z) + 2.2))
    ax.barh(z.index, z.values, height=0.62,
            color=[DIV(0.5 + 0.5 * np.clip(v / 2.2, -1, 1)) for v in z.values])
    ax.axvline(0, color=INK_2, lw=1)
    for y, v in enumerate(z.values):
        ax.text(v + (0.05 if v >= 0 else -0.05), y, f"{v:+.2f}", va="center", fontsize=8.5,
                color=INK_2, ha="left" if v >= 0 else "right")
    lim = max(np.abs(z).max() * 1.35, 1)
    ax.set_xlim(-lim, lim)
    ax.set_xlabel("Độ lệch chuẩn so với mặt bằng ngành (dương = tốt hơn)")
    ax.grid(axis="y", visible=False)
    frame(ax, f"{NAME[bank_key]} — mạnh yếu ở đâu so với mặt bằng",
          f"Nhóm {NHOM[bank_key]} · điểm tổng hợp thô {diem_tho[bank_key]:+.2f} · "
          f"hạng {list(diem_tho.index[::-1]).index(bank_key) + 1}/{len(diem_tho)}",
          "Mọi tiêu chí đã nhân chiều nên thanh phải luôn là tốt. Tiêu chí thiếu số liệu bị bỏ khỏi "
          "hồ sơ này chứ không tính bằng 0.")
    save(fig, f"fig_29_ho_so_{bank_key}")
    plt.show()


ve_ho_so(HO_SO)
# ## 12. Bảng tổng hợp — bản đọc được bằng số
#
# Mỗi biểu đồ ở trên đều nên có một "bản song sinh" dạng bảng: người không phân biệt được màu,
# người in đen trắng, và người cần con số chính xác đều dùng bảng này. Đây cũng là đầu vào tự
# nhiên cho bước xếp hạng TOPSIS/AHP ở giai đoạn sau.

In [ ]:
th = pd.DataFrame({"bank_key": LISTED})
th["Ngân hàng"] = th["bank_key"].map(NAME)
th["Mã"] = th["bank_key"].map(TICKER)
th = th.merge(rank[["bank_key", "rate"]].rename(columns={"rate": f"LS gửi {DEP_TERM}T"}), how="left")
th = th.merge(lend_last[["bank_key", "lending_avg_percent", "month"]]
              .rename(columns={"lending_avg_percent": "LS vay", "month": "Tháng công bố"}), how="left")

qL = fact_quarter[fact_quarter["period"] == Q].set_index("bank_key")
th["NIM (TTM)"] = th["bank_key"].map(qL["nim_ttm_percent"])
th["NPL"] = th["bank_key"].map(qL["npl_percent"])
th["LDR"] = th["bank_key"].map(qL["ldr_computed_percent"])
th["Dư nợ (nghìn tỷ)"] = th["bank_key"].map(qL["loans_ty_vnd"] / NGHIN_TY)
th["Điểm app"] = th["bank_key"].map(now)
th["Điểm giao dịch"] = th["bank_key"].map(net["tong"])

th = (th.drop(columns="bank_key").sort_values(f"LS gửi {DEP_TERM}T", ascending=False)
      .reset_index(drop=True))
th.index += 1

print(f"Lãi suất tiền gửi: {DEP_SOURCE} · {DEP_TERM} tháng · {DEP_CHANNEL} · {SNAP_TXT}")
print(f"Chỉ tiêu tài chính: quý {Q} · Điểm app: bình quân trọng số 12 tháng gần nhất")
th.round(2)

## 13. Ghi chú khi dùng lại notebook này

**Đổi lát cắt.** Ba biến ở mục 5 (`DEP_SOURCE`, `DEP_TERM`, `DEP_CHANNEL`) và `BANK` ở mục 6.3
là toàn bộ "slicer" — đổi giá trị rồi chạy lại từ ô đó trở xuống, mọi hình và bảng tổng hợp đi
theo. Chọn `DEP_SOURCE = "laisuattietkiem"` thì nhớ nó chỉ có 17/22 mã và riêng ô 12 tháng/online
chỉ có 12 mã — số ngân hàng trên hình sẽ tụt xuống, đó là dữ liệu chứ không phải lỗi.

**Làm mới dữ liệu.** Notebook đọc `data/powerbi/` chứ không đọc `data/` gốc. Sau khi cào thêm,
chạy lại `python export_powerbi.py` rồi chạy lại notebook từ đầu.

**Nối sang TOPSIS/AHP.** Mục 11 để lại bốn biến dùng được ngay ở giai đoạn sau:
`MT` (ma trận quyết định 22 × 16), `CHIEU` (chiều từng tiêu chí), `QUY_CHIEU` (bảng mốc tứ phân
vị + ngưỡng tốt) và `Z` (ma trận đã chuẩn hóa, dương = tốt). Muốn thêm/bớt tiêu chí thì sửa
đúng một chỗ — danh sách `TIEU_CHI` — mọi hình và bảng trong mục đó đi theo.

**Ảnh xuất ra.** `SAVE_FIG = True` ghi PNG 200 dpi vào `figures/`, đặt tên `fig_NN_*.png` theo
đúng thứ tự trong notebook để chèn thẳng vào bản Word/LaTeX của khóa luận.

**Phân biệt ngân hàng.** Ba tầng ở mục 2: màu theo nhóm (3 màu), tên viết thẳng trên hình, và
lưới nhỏ mỗi ngân hàng một ô. Cần một ngân hàng cụ thể thì tra lưới nhỏ (mục 5.3, 6.2, 8.2, 9.1, 10)
hoặc bảng tổng hợp ở mục 12 — đừng cố dò màu trên biểu đồ 22 đường.

**Sáu thứ cố ý không vẽ:**

- **CAR** — `fact_bank_quarterly.car_percent` độ đầy 11,8%; bản trong `fact_financial`
  (`RT_BANK_CAR`) nhìn thì đủ 654 dòng nhưng **564 dòng bằng đúng 0**, còn tệ hơn vì số 0 giả
  danh dữ liệu. Cả hai đều không vẽ được.
- **P/E** — có giá trị nhưng chạy từ −1.015 đến 937.746; dùng P/B thay thế (mục 8.5).
- **EPS, BVPS, chi phí hoạt động, dòng tiền tài chính** — rỗng hoàn toàn trong `fact_financial`.
- **Lãi suất cho vay dạng đường** — mỗi ngân hàng chỉ có 1–15 tháng, vẽ đường thì đa số thành
  một chấm rời. Chuỗi dài thay thế là lợi suất tài sản sinh lời (mục 6.2), và nó **không phải**
  lãi suất cho vay.
- **Biểu đồ hai trục y** — chỗ README gợi ý hai trục (lãi suất điều hành và CPI) được tách thành
  hai panel dùng chung trục thời gian ở mục 4.1. ROE và ROA ở mục 8.1 cũng vậy: hai panel
  cạnh nhau chứ không phải hai trục.
- **Trục thời gian dạng phân loại** — cả lãi suất tiền gửi (27 ngày chụp rải rác) lẫn chuỗi quý
  của từng ngân hàng đều giữ khoảng cách thật giữa các mốc; quý không có số liệu để trống chứ
  không dồn lại cho liền.